# 🏀 March Machine Learning Mania 2026 
**Author: Karl Estampador**

## 1. Setup & Installation

In [2]:
# %%capture
# !pip install statsmodels scikit-learn pandas numpy lightgbm optuna

**Import Libraries**:

We are using NumPy and Pandas for data manipulation, scikit-learn for PCA, imputation, logistic regression, cross-validation, & isotonic calibration,  `SimpleExpSmoothing` from statsmodels, and `lightgbm` for gradient boosted tree modelling.

In [3]:
import os, json, asyncio, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer  
from sklearn.isotonic import IsotonicRegression 
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
import lightgbm as lgb  
import optuna 

warnings.filterwarnings('ignore')

print('✅ All imports successful — we are ready!')

✅ All imports successful — we are ready!


**Manually recorded best hyperparameters:**

Below are the best hyperparameters recorded in `best_params.json`. These are updated in two ways: manually here in Cell 6 (as a fallback), and automatically by the optimizer which overwrites this file on completion.

**Current values** reflect Stage 1 results recovered from the overnight differential evolution run (eval 352, Brier=0.17595). Stage 2 (Optuna) will overwrite these with the full set of model parameters once it completes.

In [4]:
import json
best_params = {
    # LR: Elo + stats (Differential Evolution, 260 evals, opt Brier=0.17595)
    "ELO_K":               19.3,
    "ELO_HCA":             112.3,
    "ELO_RECENCY_ALPHA":   0.028,
    "STATS_SMOOTH_ALPHA":  0.186,

    # LGB: model params (Optuna TPE, 1559 trials, opt Brier=0.17552)
    "LOGREG_C":            9.92352274686263,
    "ENSEMBLE_WEIGHT":     0.3859610723970893,
    "LGB_N_ESTIMATORS":    331,
    "LGB_LEARNING_RATE":   0.011577891353652614,
    "LGB_MAX_DEPTH":       3,
    "LGB_NUM_LEAVES":      49,
    "LGB_REG_ALPHA":       0.3641094597731368,
    "LGB_REG_LAMBDA":      2.875129050910777,
    "LGB_SUBSAMPLE":       0.9471763428357364,
    "LGB_COLSAMPLE":       0.6465419607066056,

    # Diagnostics
    "s1_opt_brier":        0.17595,
    "s2_opt_brier":        0.17552,
    "s2_holdout_brier":    0.16341,
}

if 'PARAMS_PATH' not in globals():
    PARAMS_PATH = 'best_params.json'

with open(PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=2)
print("✅ best_params.json written!")

✅ best_params.json written!


**Load hyperparameters & initialize global configuration:**

`_load_params()` reads `best_params.json` if it exists, and if not, falls back to conservative defaults. Saved values are merged over defaults, so new parameters added in future iterations don't cause KeyErrors on old JSON files.

The global state dicts (`ELO`, `STATS`, `MASSEY`, `MASSEY_PCA`, `MASSEY_PCA_FOLDS`) are initialized empty here and populated by the `compute_*` functions in Section 4.

Note to self: Three new globals are added: `LGB_MODEL` holds the fitted LightGBM classifier, `CALIBRATOR` holds the isotonic regression calibrator fit on out-of-fold blended predictions (C2), and `ENSEMBLE_WEIGHT` controls the blend ratio at inference time. `LGB_PARAMS` stores LightGBM hyperparameters in one place so they stay consistent across the final fit and the walk-forward folds.

In [5]:
import json as _json, os as _os

# Default hyperparameters used on a fresh kernel with no saved file.
_DEFAULTS = dict(
    ELO_K             = 10.0,
    ELO_HCA           = 100.0,
    ELO_RECENCY_ALPHA = 0.3,
    STATS_SMOOTH_ALPHA= 0.3,
    LOGREG_C          = 1.0,
)

PARAMS_PATH = 'best_params.json'

def _load_params():
    """Load saved params if they exist, otherwise return defaults."""
    if _os.path.exists(PARAMS_PATH):
        with open(PARAMS_PATH) as _f:
            saved = _json.load(_f)
        # Saved values override defaults (handles new params added later)
        merged = {**_DEFAULTS, **saved}
        print(f'✅ Loaded optimised hyperparameters from {PARAMS_PATH}')
        for k, v in merged.items():
            print(f'   {k:22s} = {v}')
        return merged
    else:
        print('ℹ️  No best_params.json found — using default hyperparameters.')
        print('   Run run_hyperparameter_optimization() to find optimal values.')
        return dict(_DEFAULTS)

_p = _load_params()

# CONFIGURATION
DATA_DIR   = 'data'
CURRENT_SEASON = 2026
ELO_INIT   = 1500   # not optimized, doesn't improve model if it is
ELO_REGRESS_ALPHA = 0.75

# unpacks optimized hyperparameters into globals
ELO_K              = _p['ELO_K']
ELO_HCA            = _p['ELO_HCA']
ELO_RECENCY_ALPHA  = _p['ELO_RECENCY_ALPHA']
STATS_SMOOTH_ALPHA = _p['STATS_SMOOTH_ALPHA']
LOGREG_C           = _p['LOGREG_C']

USE_MASSEY_PCA = False

# GLOBAL STATE DICTS
DATA             = {}
ELO              = {} # {(season, team_id): elo_rating}
STATS            = {} # {(season, team_id): {stat: value}}
MASSEY           = {} # mean percentile per team-season
MASSEY_PCA       = {} # (PC1, PC2) per team-season, all-season fit
MASSEY_PCA_FOLDS = {} # {eval_season: {(season, team_id): (pc1, pc2)}}
MODEL            = None

# LightGBM ensemble globals
LGB_MODEL        = None   # fitted LightGBM classifier
CALIBRATOR       = None   # isotonic regression calibrator (fit on OOF blended preds)
ENSEMBLE_WEIGHT  = _p.get('ENSEMBLE_WEIGHT', 0.5)   # loaded from JSON if available

# LightGBM hyperparameters — loaded from best_params.json when available,
# otherwise sensible defaults. Updated by run_hyperparameter_optimization().
LGB_PARAMS = dict(
    n_estimators     = int(_p.get('LGB_N_ESTIMATORS',  300)),
    learning_rate    =     _p.get('LGB_LEARNING_RATE', 0.05),
    max_depth        = int(_p.get('LGB_MAX_DEPTH',       4)),
    num_leaves       = int(_p.get('LGB_NUM_LEAVES',     15)),
    reg_alpha        =     _p.get('LGB_REG_ALPHA',     1.0),
    reg_lambda       =     _p.get('LGB_REG_LAMBDA',    1.0),
    subsample        =     _p.get('LGB_SUBSAMPLE',     0.8),
    colsample_bytree =     _p.get('LGB_COLSAMPLE',     0.8),
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

✅ Loaded optimised hyperparameters from best_params.json
   ELO_K                  = 19.3
   ELO_HCA                = 112.3
   ELO_RECENCY_ALPHA      = 0.028
   STATS_SMOOTH_ALPHA     = 0.186
   LOGREG_C               = 9.92352274686263
   ENSEMBLE_WEIGHT        = 0.3859610723970893
   LGB_N_ESTIMATORS       = 331
   LGB_LEARNING_RATE      = 0.011577891353652614
   LGB_MAX_DEPTH          = 3
   LGB_NUM_LEAVES         = 49
   LGB_REG_ALPHA          = 0.3641094597731368
   LGB_REG_LAMBDA         = 2.875129050910777
   LGB_SUBSAMPLE          = 0.9471763428357364
   LGB_COLSAMPLE          = 0.6465419607066056
   s1_opt_brier           = 0.17595
   s2_opt_brier           = 0.17552
   s2_holdout_brier       = 0.16341


## 2. Loading Data

In [6]:
m_teams = pd.read_csv(f'{DATA_DIR}/MTeams.csv')
w_teams = pd.read_csv(f'{DATA_DIR}/WTeams.csv')

m_regular = pd.read_csv(f'{DATA_DIR}/MRegularSeasonDetailedResults.csv')
w_regular = pd.read_csv(f'{DATA_DIR}/WRegularSeasonDetailedResults.csv')
 
m_tourney = pd.read_csv(f'{DATA_DIR}/MNCAATourneyDetailedResults.csv')
w_tourney = pd.read_csv(f'{DATA_DIR}/WNCAATourneyDetailedResults.csv')

# men's only - no women's equivalent available
m_massey = pd.read_csv(f'{DATA_DIR}/MMasseyOrdinals.csv')

m_seeds = pd.read_csv(f'{DATA_DIR}/MNCAATourneySeeds.csv')
w_seeds = pd.read_csv(f'{DATA_DIR}/WNCAATourneySeeds.csv')

print(f'✅ Data loaded. Massey Ordinals: {m_massey.shape[0]:,} rows, ')
print(f'   seasons {m_massey["Season"].min()}–{m_massey["Season"].max()}, ')
print(f'   {m_massey["SystemName"].nunique()} unique ranking systems.')

✅ Data loaded. Massey Ordinals: 5,865,001 rows, 
   seasons 2003–2026, 
   197 unique ranking systems.


In [7]:
# combines men's and women's data into the same DataFrames
regular_results = pd.concat([m_regular, w_regular])
tourney_results = pd.concat([m_tourney, w_tourney])
seeds = pd.concat([m_seeds, w_seeds])

**We filter all datasets to 2003 and later**

This is because detailed box-score statistics are only available from 2003 onward, so we filter to keep the feature set consistent across all training samples.

In [8]:
season = 2003 
regular_results = regular_results.loc[regular_results["Season"] >= season]
tourney_results = tourney_results.loc[tourney_results["Season"] >= season]
seeds = seeds.loc[seeds["Season"] >= season]

## 3. Exploring Data

In [9]:
regular_results

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87182,2026,131,3471,60,3218,48,N,0,25,69,...,16,8,8,2,31,9,17,3,6,15
87183,2026,132,3158,68,3220,56,N,0,23,61,...,21,3,5,9,25,10,13,2,4,17
87184,2026,132,3192,79,3254,57,H,0,31,60,...,20,12,14,5,19,2,11,5,4,15
87185,2026,132,3221,77,3250,70,H,0,31,60,...,24,9,12,8,17,17,11,6,2,12


In [10]:
tourney_results

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
956,2025,147,3163,78,3425,64,N,0,27,60,...,13,23,26,9,26,7,15,8,2,18
957,2025,147,3400,58,3395,47,N,0,24,61,...,20,19,21,9,21,8,16,1,3,14
958,2025,151,3163,85,3417,51,N,0,33,60,...,16,7,8,6,20,11,18,5,2,12
959,2025,151,3376,74,3400,57,N,0,29,57,...,10,9,12,8,18,10,14,6,5,17


## 4. Feature Engineering

**Define Elo rating computation with recency weighting and margin-of-victory (MOV) scaling.**

Two key modifications over standard Elo:

1. **Recency weighting**: The K-factor grows exponentially as the season progresses, so later games carry more weight, reflecting that late-season form is more predictive of tournament performance.

2. **FiveThirtyEight MOV multiplier**: K is further scaled by `log(margin + 1) × autocorrelation_correction`. This gives larger wins more Elo movement with diminishing returns, while the autocorrelation correction prevents strong teams from snowballing ratings by blowing out weak opponents.

> **Important**: `_opt_compute_elo()` in the optimizer cell (Section 5) must mirror this formula exactly. If they diverge, the optimizer will tune `ELO_K` for the wrong function and `best_params.json` will be silently incorrect.

In [11]:
# margin of victory (MOV) multiplier - FiveThirtyEight approach
def mov_multiplier(margin: float, winner_elo_diff: float) -> float:
    """
    FiveThirtyEight MOV multiplier applied to the Elo K-factor.

    Formula:
        log_term      = log(margin + 1)           # diminishing returns on blowouts
        ac_correction = 2.2 / (winner_elo_diff * 0.001 + 2.2)  # anti-snowball
        return log_term * ac_correction

    The autocorrelation correction prevents heavy favourites from
    inflating their ratings by blowing out weak opponents.
    """
    log_term     = np.log(margin + 1)
    ac_correction = 2.2 / (winner_elo_diff * 0.001 + 2.2)
    return log_term * ac_correction

def compute_elo_ratings() -> dict:
    """Compute Elo ratings with exponential recency weighting."""

    def _run_elo(regular_df, tourney_df):
        elo = {}
        season_elos = {}

        # Combine regular + tourney; process chronologically
        all_games = pd.concat([regular_df, tourney_df]).sort_values(['Season', 'DayNum'])
        day_ranges = (
            all_games.groupby('Season')['DayNum']
            .agg(min_day='min', max_day='max')
        )

        prev_season = None
        for _, row in all_games.iterrows():
            season = row['Season']

            # Season boundary: snapshot current Elo, then regress toward mean
            if season != prev_season and prev_season is not None:
                for tid, r in elo.items():
                    season_elos[(prev_season, tid)] = r
                elo = {tid: ELO_REGRESS_ALPHA * r + (1 - ELO_REGRESS_ALPHA) * ELO_INIT for tid, r in elo.items()}
            prev_season = season

            # Recency weight: games later in season count more
            min_d = day_ranges.loc[season, 'min_day']
            max_d = day_ranges.loc[season, 'max_day']
            t = (row['DayNum'] - min_d) / max(max_d - min_d, 1) # t in [0,1]
            K_game = ELO_K * np.exp(ELO_RECENCY_ALPHA * t)  

            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo = elo.get(w_id, ELO_INIT)
            l_elo = elo.get(l_id, ELO_INIT)

            # home court adjustment
            w_loc = row.get('WLoc', 'N')
            w_adj = w_elo + (ELO_HCA if w_loc == 'H' else (-ELO_HCA if w_loc == 'A' else 0))

            exp_w = 1.0 / (1.0 + 10 ** ((l_elo - w_adj) / 400.0))

            # scale K by MOV before updating ratings
            margin          = row['WScore'] - row['LScore']
            winner_elo_diff = w_adj - l_elo   # positive = winner was favoured
            K_game         *= mov_multiplier(margin, winner_elo_diff)

            elo[w_id] = w_elo + K_game * (1.0 - exp_w)
            elo[l_id] = l_elo + K_game * (0.0 - (1.0 - exp_w))

        if prev_season:
            for tid, r in elo.items():
                season_elos[(prev_season, tid)] = r
        return season_elos

    m_elos = _run_elo(m_regular, m_tourney)
    w_elos = _run_elo(w_regular, w_tourney)
    ELO.update(m_elos)
    ELO.update(w_elos)

    m_names = dict(zip(m_teams['TeamID'], m_teams['TeamName']))
    w_names = dict(zip(w_teams['TeamID'], w_teams['TeamName']))
    latest_m = max(s for s, _ in m_elos.keys())
    latest_w = max(s for s, _ in w_elos.keys())
    top_m = sorted([(tid, r) for (s, tid), r in m_elos.items() if s == latest_m], key=lambda x: -x[1])[:5]
    top_w = sorted([(tid, r) for (s, tid), r in w_elos.items() if s == latest_w], key=lambda x: -x[1])[:5]

    return {
        'status': 'success',
        'total_ratings': len(ELO),
        'top_mens':   [f"{m_names.get(t, t)}: {r:.0f}" for t, r in top_m],
        'top_womens': [f"{w_names.get(t, t)}: {r:.0f}" for t, r in top_w],
        'message': f'Recency-weighted Elo computed through {latest_m} (men) and {latest_w} (women).'
    }

print('✅ compute_elo_ratings defined (recency weighting + MOV multiplier)')

✅ compute_elo_ratings defined (recency weighting + MOV multiplier)


**Define exponentially smoothed team stats computation.**

For each team-season, per-game box-score values are collected in chronological order and smoothed using Simple Exponential Smoothing (SES): `s[t] = α × x[t] + (1 - α) × s[t-1]`. The final smoothed value captures the team's end-of-regular-season form.

Stats computed:

| Group | Stats |
|---|---|
| **Traditional** | field-goal %, 3-point %, free-throw %, total rebounds, turnovers, assists |
| **Pace-adjusted efficiency** | `off_eff` (points scored per 100 possessions), `def_eff` (points allowed per 100 possessions), `net_rating` = `off_eff − def_eff` |
| **Dean Oliver's Four Factors (offensive)** | `efg_pct` (effective FG%), `tov_pct` (turnover rate), `orb_pct` (offensive rebound rate), `ftr` (free-throw rate) |
| **Dean Oliver's Four Factors (defensive)** | `def_efg_pct`, `def_tov_pct`, `def_orb_pct`, `def_ftr` — the same four factors computed from the opponent's perspective |

Pace-adjusted efficiency uses Dean Oliver's possession formula: `poss ≈ FGA − OReb + TOV + 0.44 × FTA`. Without pace adjustment, raw totals confound fast-paced teams with efficient ones. The Four Factors explain ~96–98% of the variance in offensive efficiency and directly map to how modern analytics evaluate team quality.

Both the team's *and* the opponent's box-score columns are included in each game row so that `def_eff` and the defensive Four Factors can be computed correctly on a per-game basis before smoothing.

The SES is implemented as a direct NumPy recurrence (~100× faster than statsmodels), which matters when the optimizer calls this function ~400+ times.

In [12]:
#     Note: WRegularSeasonDetailedResults box-score data is available from
#     2010 onward on Kaggle. Women's team-seasons before 2010 will have no
#     STATS entry and fall back to 0.0 diffs in _get_stats_diff — exactly the
#     same graceful degradation as men's teams before 2003.

def _safe_ses(values, alpha=None):
    """
    Compute Simple Exponential Smoothing and return the final smoothed value.
    Uses a fast numpy recurrence instead of statsmodels (~100x faster).
    alpha defaults to None so it reads STATS_SMOOTH_ALPHA at call time,
    ensuring hyperparameter updates take effect without redefining the function.
    """
    if alpha is None:
        alpha = STATS_SMOOTH_ALPHA
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0: return np.nan
    if len(arr) == 1: return float(arr[0])
    s = arr[0] # initialise at first observation
    for x in arr[1:]:
        s = alpha * x + (1.0 - alpha) * s
    return float(s)


def compute_smoothed_stats() -> dict:
    """
    For each team-season, apply SES to per-game box-score stats in
    chronological order and store the end-of-season smoothed value.

    Stats computed for BOTH men's and women's teams (17 stats each):
      Traditional  : fg_pct, fg3_pct, ft_pct, reb, to, ast
      A1 efficiency: off_eff, def_eff, net_rating
      A2 Four Fac. : efg_pct, tov_pct, orb_pct, ftr            (offensive)
                     def_efg_pct, def_tov_pct, def_orb_pct,
                     def_ftr                                    (defensive)

    Possession formula (Dean Oliver):
        poss ≈ FGA − OReb + TOV + 0.44 × FTA
    The 0.44 coefficient accounts for free-throw trips that do not end
    a possession (and-ones, technicals, etc.).

    Both team and opponent box-score columns are stacked so that def_eff
    and the defensive Four Factors can be computed on each game row before
    exponential smoothing is applied.
    """

    def _process(regular_df):
        shared = ['Season', 'DayNum']

        #  Winner perspective 
        # Include opponent (loser) box-score columns for A1/A2 defensive stats
        w_view = regular_df[shared + [
            'WTeamID', 'WScore', 'LScore',
            'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
            'WOR', 'WDR', 'WTO', 'WAst',
            # opponent (loser) stats — needed for opp_poss + defensive Four Factors
            'LFGM', 'LFGA', 'LFGM3', 'LFTA', 'LOR', 'LDR', 'LTO',
        ]].copy()
        w_view.columns = shared + [
            'TeamID', 'Score', 'OppScore',
            'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'TO', 'Ast',
            'OppFGM', 'OppFGA', 'OppFGM3', 'OppFTA', 'OppOR', 'OppDR', 'OppTO',
        ]

        # Loser perspective 
        l_view = regular_df[shared + [
            'LTeamID', 'LScore', 'WScore',
            'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
            'LOR', 'LDR', 'LTO', 'LAst',
            # opponent (winner) stats
            'WFGM', 'WFGA', 'WFGM3', 'WFTA', 'WOR', 'WDR', 'WTO',
        ]].copy()
        l_view.columns = shared + [
            'TeamID', 'Score', 'OppScore',
            'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'TO', 'Ast',
            'OppFGM', 'OppFGA', 'OppFGM3', 'OppFTA', 'OppOR', 'OppDR', 'OppTO',
        ]

        games = pd.concat([w_view, l_view], ignore_index=True)

        # Traditional per-game stats
        games['fg_pct']  = games['FGM']  / games['FGA'].replace(0, np.nan)
        games['fg3_pct'] = games['FGM3'] / games['FGA3'].replace(0, np.nan)
        games['ft_pct']  = games['FTM']  / games['FTA'].replace(0, np.nan)
        games['reb']     = games['OR'] + games['DR']
        games['to']      = games['TO']
        games['ast']     = games['Ast']

        # A1: Pace-adjusted efficiency
        # Oliver possession formula: poss ≈ FGA − OReb + TOV + 0.44 × FTA
        poss     = (games['FGA'] - games['OR'] + games['TO']
                    + 0.44 * games['FTA'])
        opp_poss = (games['OppFGA'] - games['OppOR'] + games['OppTO']
                    + 0.44 * games['OppFTA'])

        games['off_eff']    = (games['Score']    / poss.replace(0, np.nan)) * 100
        games['def_eff']    = (games['OppScore'] / opp_poss.replace(0, np.nan)) * 100
        games['net_rating'] = games['off_eff'] - games['def_eff']  # ✨ replaces net_eff

        # A2: Dean Oliver's Four Factors (offensive)
        games['efg_pct'] = (
            (games['FGM'] + 0.5 * games['FGM3'])
            / games['FGA'].replace(0, np.nan)
        )
        games['tov_pct'] = games['TO'] / (
            games['FGA'] + 0.44 * games['FTA'] + games['TO']
        ).replace(0, np.nan)
        games['orb_pct'] = games['OR'] / (
            games['OR'] + games['OppDR']
        ).replace(0, np.nan)
        games['ftr'] = games['FTA'] / games['FGA'].replace(0, np.nan)

        # A2: Dean Oliver's Four Factors (defensive)
        games['def_efg_pct'] = (
            (games['OppFGM'] + 0.5 * games['OppFGM3'])
            / games['OppFGA'].replace(0, np.nan)
        )
        games['def_tov_pct'] = games['OppTO'] / (
            games['OppFGA'] + 0.44 * games['OppFTA'] + games['OppTO']
        ).replace(0, np.nan)
        games['def_orb_pct'] = games['OppOR'] / (
            games['OppOR'] + games['DR']
        ).replace(0, np.nan)
        games['def_ftr'] = games['OppFTA'] / games['OppFGA'].replace(0, np.nan)

        stat_cols = [
            # traditional
            'fg_pct', 'fg3_pct', 'ft_pct', 'reb', 'to', 'ast',
            # A1: pace-adjusted efficiency
            'off_eff', 'def_eff', 'net_rating',
            # A2: Four Factors (offensive + defensive)
            'efg_pct', 'tov_pct', 'orb_pct', 'ftr',
            'def_efg_pct', 'def_tov_pct', 'def_orb_pct', 'def_ftr',
        ]
        games = games.sort_values(['Season', 'TeamID', 'DayNum'])
        smoothed = {}
        for (season, team_id), grp in games.groupby(['Season', 'TeamID']):
            row = {}
            for col in stat_cols:
                series = grp[col].dropna().tolist()
                row[col] = _safe_ses(series) if series else np.nan
            smoothed[(season, team_id)] = row
        return smoothed

    m_stats = _process(m_regular)
    w_stats = _process(w_regular)
    STATS.update(m_stats)
    STATS.update(w_stats)

    total = len(STATS)
    sample_key = next(iter(STATS))
    sample_val = STATS[sample_key]
    return {
        'status': 'success',
        'total_team_seasons': total,
        'sample_key': str(sample_key),
        'sample_stats': {
            k: f'{v:.4f}' for k, v in sample_val.items()
            if not (isinstance(v, float) and np.isnan(v))
        },
        'message': (
            f'SES stats computed for {total} team-seasons '
            f'with alpha={STATS_SMOOTH_ALPHA:.4f}. '
            f'Stats per team-season: {len(sample_val)} '
            f'(6 traditional + 3 A1 efficiency + 8 A2 Four Factors).'
        )
    }

print('✅ compute_smoothed_stats defined (A1: off/def/net_rating | A2: 8 Four Factors | E1: women\'s derived efficiency)')

✅ compute_smoothed_stats defined (A1: off/def/net_rating | A2: 8 Four Factors | E1: women's derived efficiency)


**E1 — Women's Derived Efficiency Metrics: Diagnostic.**

The cell below verifies that `compute_smoothed_stats` is correctly populating advanced stats for women's teams. It reports:

- Total team-seasons in `STATS` broken down by gender (men vs. women)
- The range of seasons for which women's stats are available
- Sample smoothed values for a recent women's team (as a sanity check)
- Coverage rate: the fraction of women's tournament matchups (2010–2025) for which both teams have at least one advanced stat populated

This diagnostic should be run once after `compute_smoothed_stats()` is called in Section 6. A coverage rate below ~90% would indicate a data-loading issue that needs investigation.

In [13]:
# E1 DIAGNOSTIC — Women's Derived Efficiency Metrics coverage check
# Verifies that compute_smoothed_stats() correctly populated STATS for
# women's teams with A1 (off_eff, def_eff, net_rating) and A2 (Four Factors).
# Run this after the pipeline cell (Section 6) has executed.

def run_e1_diagnostic():
    if not STATS:
        print('⚠️  STATS is empty — run the pipeline cell (Section 6) first.')
        return

    # Men's TeamIDs are < 3000; Women's TeamIDs are >= 3000 (Kaggle convention)
    W_ID_THRESHOLD = 3000

    m_keys = [(s, t) for (s, t) in STATS if t < W_ID_THRESHOLD]
    w_keys = [(s, t) for (s, t) in STATS if t >= W_ID_THRESHOLD]

    print('=' * 60)
    print('E1 — Women\'s Derived Efficiency Stats: Diagnostic')
    print('=' * 60)
    print(f'  Men\'s  team-seasons in STATS : {len(m_keys):>6,}')
    print(f'  Women\'s team-seasons in STATS: {len(w_keys):>6,}')

    if not w_keys:
        print('\n⚠️  No women\'s team-seasons found in STATS.')
        print('   Check that w_regular was loaded and compute_smoothed_stats() ran.')
        return

    w_seasons = sorted(set(s for s, _ in w_keys))
    print(f'\n  Women\'s seasons with stats : {w_seasons[0]}–{w_seasons[-1]} ({len(w_seasons)} seasons)')

    # Sample one recent women's team for a sanity-check readout
    recent_w = sorted(w_keys, reverse=True)[0]
    sample = STATS[recent_w]
    print(f'\n  Sample women\'s team-season : Season={recent_w[0]}, TeamID={recent_w[1]}')
    print(f'  {"Stat":<20} {"Value":>10}')
    print(f'  {"-"*32}')
    key_stats = [
        ('off_eff',       'Offensive Eff (A1)'),
        ('def_eff',       'Defensive Eff (A1)'),
        ('net_rating',    'Net Rating (A1)'),
        ('efg_pct',       'eFG% (A2)'),
        ('tov_pct',       'TOV% (A2)'),
        ('orb_pct',       'ORB% (A2)'),
        ('ftr',           'FTR (A2)'),
        ('def_efg_pct',   'Def eFG% (A2)'),
        ('def_tov_pct',   'Def TOV% (A2)'),
        ('def_orb_pct',   'Def ORB% (A2)'),
        ('def_ftr',       'Def FTR (A2)'),
    ]
    for key, label in key_stats:
        val = sample.get(key, float('nan'))
        flag = '' if not (isinstance(val, float) and __import__('math').isnan(val)) else '  ← ⚠️ NaN'
        print(f'  {label:<20} {val:>10.4f}{flag}')

    # Coverage: what fraction of women's tourney matchups (2010+) have both teams in STATS?
    w_stat_team_seasons = set(w_keys)
    covered, total = 0, 0
    for _, row in w_tourney[w_tourney['Season'] >= 2010].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']
        total += 1
        # _get_stats_diff uses season-1 stats
        if (season - 1, w_id) in w_stat_team_seasons and (season - 1, l_id) in w_stat_team_seasons:
            covered += 1

    coverage_pct = 100 * covered / total if total > 0 else 0
    status = '✅' if coverage_pct >= 80 else '⚠️ '
    print(f'\n  Women\'s tourney coverage (2010–2025):')
    print(f'  {status} {covered}/{total} matchups have stats for both teams ({coverage_pct:.1f}%)')
    if coverage_pct < 80:
        print('  → Coverage below 80%. Consider checking w_regular season range.')
    elif coverage_pct < 95:
        print('  → Some early seasons may fall back to 0-diff (expected for pre-2010 data).')
    else:
        print('  → Excellent coverage. E1 is fully operational for the women\'s model.')
    print('=' * 60)

# This diagnostic runs automatically only if STATS has been populated.
# If STATS is empty (fresh kernel before running Section 6), it will
# print a reminder — this is safe to leave in the notebook.
run_e1_diagnostic()


⚠️  STATS is empty — run the pipeline cell (Section 6) first.


**Define Massey Ordinals feature engineering (Phase 1 + Phase 2 PCA).**

The Massey Ordinals dataset contains rankings from ~196 independent systems (Sagarin, Pomeroy, RPI, etc.) — each measuring the same latent variable (team quality) with noise.

**Phase 1** (`compute_massey_features`): takes the simple mean percentile across all systems.

**Phase 2** (`compute_massey_pca` / `compute_massey_pca_folds`): applies PCA to the team × system percentile matrix.
- **PC1** captures the consensus strength signal and typically explains 70–90% of variance.
- **PC2** captures a secondary dimension — often a style/tempo split between offense-heavy systems (Pomeroy) and schedule-strength systems.

**Leakage prevention**: Walk-forward validation uses `compute_massey_pca_folds()`, which fits PCA on seasons before the eval season and only *transforms* the eval season — ensuring future rankings never influence past predictions.

In [14]:
# Phase 1 helper: percentile normalisation 
def _build_massey_percentile_df():
    """
    Filter Massey data to final pre-tournament rankings (RankingDayNum=133)
    and normalise each system-season to a percentile in [0, 1]:
        percentile = 1 - (rank - 1) / (N - 1)
    Rank #1 -> 1.0 (best), rank #N -> 0.0 (worst).
    clip(lower=1) guards the degenerate single-team-system edge case.
    """
    df = m_massey[m_massey['RankingDayNum'] == 133].copy()
    df['N'] = df.groupby(['Season', 'SystemName'])['OrdinalRank'].transform('count')
    df['percentile'] = 1.0 - (df['OrdinalRank'] - 1) / (df['N'] - 1).clip(lower=1)
    return df


#  Phase 1: Mean percentile across all systems
def compute_massey_features() -> dict:
    """Compute mean percentile rank across all Massey systems; store in MASSEY."""
    df = _build_massey_percentile_df()
    massey_scores = (
        df.groupby(['Season', 'TeamID'])['percentile']
        .mean()
        .reset_index()
        .rename(columns={'percentile': 'mean_pct'})
    )
    MASSEY.clear()
    for _, row in massey_scores.iterrows():
        MASSEY[(int(row['Season']), int(row['TeamID']))] = float(row['mean_pct'])

    latest_season = df['Season'].max()
    n_sys_latest  = df[df['Season'] == latest_season]['SystemName'].nunique()
    seasons_list  = sorted(df['Season'].unique().tolist())
    sample_pcts   = [v for (s, _), v in MASSEY.items() if s == latest_season]
    mean_pct      = sum(sample_pcts) / len(sample_pcts) if sample_pcts else float('nan')
    return {
        'status'                  : 'success',
        'total_team_seasons'      : len(MASSEY),
        'seasons_covered'         : f'{seasons_list[0]}\u2013{seasons_list[-1]}',
        'n_seasons'               : len(seasons_list),
        'systems_in_latest_season': n_sys_latest,
        'mean_percentile_latest'  : f'{mean_pct:.4f}  (should be \u22480.50)',
        'message': (
            f'Massey mean-percentile scores computed for {len(MASSEY):,} team-seasons '
            f'across {len(seasons_list)} seasons using {n_sys_latest} systems (latest season).'
        )
    }

# original phase 1 model kept for reference, model now uses phase 2!
def _get_massey_diff(season: int, t1_id: int, t2_id: int) -> float:
    """Phase 1 mean-percentile diff (kept for reference; model uses Phase 2)."""
    import math
    m1 = MASSEY.get((season, t1_id), float('nan'))
    m2 = MASSEY.get((season, t2_id), float('nan'))
    return 0.0 if (math.isnan(m1) or math.isnan(m2)) else m1 - m2

# Phase 2 core: fit PCA on training seasons, score all seasons
def _fit_pca_and_score(train_seasons, score_seasons, pct_df):
    """
    Fit a 2-component PCA on the team x system percentile matrix built from
    train_seasons, then project teams in score_seasons into PCA space.

    PC1 captures the consensus strength signal (explains ~70-90% of variance).
    PC2 captures a secondary dimension — often a style/tempo split between
    systems like Pomeroy (offense-heavy) vs schedule-strength systems.

    Signs are arbitrary after PCA, so we flip PC1 to ensure positive values
    always mean better-ranked teams. PC2 sign is left for L2 to handle.

    Returns: dict {(season, team_id): (pc1_score, pc2_score)}
    """
    train_df = pct_df[pct_df['Season'].isin(train_seasons)]
    if len(train_df) == 0:
        return {}

    pivot_train = train_df.pivot_table(
        index=['Season', 'TeamID'],
        columns='SystemName',
        values='percentile'
    )
    pivot_train.columns.name = None

    # Fit imputer on training data (column-mean for missing values)
    imputer = SimpleImputer(strategy='mean')
    X_train = imputer.fit_transform(pivot_train.values)

    n_comp = min(2, X_train.shape[1], X_train.shape[0])
    pca = PCA(n_components=n_comp, random_state=42)
    pcs_train = pca.fit_transform(X_train)   # shape: (n_samples, n_comp)
    pc1_train = pcs_train[:, 0]
    pc2_train = pcs_train[:, 1] if n_comp >= 2 else np.zeros(len(pc1_train))

    # Ensure PC1 > 0 means better-ranked team
    mean_pct_train = (
        train_df.groupby(['Season', 'TeamID'])['percentile']
        .mean()
        .reindex(pivot_train.index)
        .values
    )
    valid = ~np.isnan(mean_pct_train)
    if valid.sum() > 1:
        corr = np.corrcoef(pc1_train[valid], mean_pct_train[valid])[0, 1]
        if corr < 0:
            pca.components_[0] *= -1   # flip only PC1's loadings
            pc1_train           *= -1

    # store training season scores as (pc1, pc2) tuples
    scores = {}
    for (season, team_id), pc1, pc2 in zip(pivot_train.index, pc1_train, pc2_train):
        scores[(int(season), int(team_id))] = (float(pc1), float(pc2))

    # transform eval seasons using training PCA (no re-fitting = no leakage)
    train_season_set = set(train_seasons)
    for season in score_seasons:
        if season in train_season_set:
            continue
        season_df = pct_df[pct_df['Season'] == season]
        if len(season_df) == 0:
            continue
        pivot_s = season_df.pivot_table(
            index=['Season', 'TeamID'],
            columns='SystemName',
            values='percentile'
        )
        pivot_s.columns.name = None
        # Align to training columns: drop unknowns, impute missing
        pivot_s = pivot_s.reindex(columns=pivot_train.columns)
        X_s = imputer.transform(pivot_s.values)
        pcs_s = pca.transform(X_s)   # shape: (n_samples, n_comp)
        pc1_s = pcs_s[:, 0]
        pc2_s = pcs_s[:, 1] if n_comp >= 2 else np.zeros(len(pc1_s))
        for (s_idx, team_id), pc1, pc2 in zip(pivot_s.index, pc1_s, pc2_s):
            scores[(int(s_idx), int(team_id))] = (float(pc1), float(pc2))

    return scores


def compute_massey_pca() -> dict:
    """
    Fit PCA on ALL available Massey seasons and store (PC1, PC2) scores in MASSEY_PCA.
    Used by train_prediction_model() for the final model trained on all
    historical data.
    """
    pct_df      = _build_massey_percentile_df()
    all_seasons = sorted(pct_df['Season'].unique().tolist())
    scores      = _fit_pca_and_score(all_seasons, all_seasons, pct_df)
    MASSEY_PCA.clear()
    MASSEY_PCA.update(scores)
    latest   = max(all_seasons)
    n_latest = sum(1 for (s, _) in MASSEY_PCA if s == latest)
    return {
        'status'            : 'success',
        'total_team_seasons': len(MASSEY_PCA),
        'seasons_covered'   : f'{all_seasons[0]}\u2013{all_seasons[-1]}',
        'teams_in_latest'   : n_latest,
        'message': (
            f'Massey PCA (PC1+PC2) scores computed for {len(MASSEY_PCA):,} team-seasons '
            f'across {len(all_seasons)} seasons (all-season fit for final model).'
        )
    }


def compute_massey_pca_folds(start=2006, end=2025) -> dict:
    """
    Pre-compute per-fold leak-free PCA scores and store in MASSEY_PCA_FOLDS.

    For each eval_season S in [start, end]:
      - PCA FIT on seasons < S only        (training seasons, no leakage)
      - Season S TRANSFORMED using that PCA (eval season)
    Result stored as MASSEY_PCA_FOLDS[S], each value a dict of
    {(season, team_id): (pc1, pc2)} tuples.
    """
    pct_df      = _build_massey_percentile_df()
    all_seasons = sorted(pct_df['Season'].unique().tolist())
    MASSEY_PCA_FOLDS.clear()
    n_folds = 0
    for eval_season in range(start, end + 1):
        train_seasons = [s for s in all_seasons if s < eval_season]
        if len(train_seasons) < 3:
            continue   # need at least 3 seasons for a meaningful PCA
        score_seasons = train_seasons + [eval_season]
        MASSEY_PCA_FOLDS[eval_season] = _fit_pca_and_score(
            train_seasons, score_seasons, pct_df
        )
        n_folds += 1
    return {
        'status'    : 'success',
        'n_folds'   : n_folds,
        'fold_range': f'{start}\u2013{end}',
        'message': (
            f'Per-fold PCA (PC1+PC2) scores pre-computed for {n_folds} eval seasons '
            f'({start}\u2013{end}). Each fold fits PCA on training seasons only.'
        )
    }


def _get_massey_pca_diff(season: int, t1_id: int, t2_id: int,
                          pca_dict: dict = None):
    """
    Return (PC1_diff, PC2_diff) tuple for a matchup (team1 - team2).

    ✨ CHANGED: returns a 2-tuple instead of a scalar.

    Positive PC1_diff -> team1 ranked higher (better) by the PCA consensus.
    PC2_diff          -> captures style/tempo dimension disagreement.
    (0.0, 0.0)        -> one/both teams missing (neutral fallback; applies
                         to women's teams, which have no Massey data).

    Args:
        season   : tournament season.
        t1_id    : TeamID of team1 (lower ID by submission convention).
        t2_id    : TeamID of team2 (higher ID by submission convention).
        pca_dict : which PCA score dict to use.
                   None (default) -> global MASSEY_PCA (all-season fit).
                   MASSEY_PCA_FOLDS[eval_season] -> leak-free per-fold scores.
    """
    if not USE_MASSEY_PCA:
        return (0.0, 0.0)
    import math
    d  = pca_dict if pca_dict is not None else MASSEY_PCA
    v1 = d.get((season, t1_id))
    v2 = d.get((season, t2_id))
    if v1 is None or v2 is None:
        return (0.0, 0.0)
    pc1_1, pc2_1 = v1
    pc1_2, pc2_2 = v2
    if math.isnan(pc1_1) or math.isnan(pc1_2):
        return (0.0, 0.0)
    pc2_diff = 0.0 if (math.isnan(pc2_1) or math.isnan(pc2_2)) else pc2_1 - pc2_2
    return (pc1_1 - pc1_2, pc2_diff)


print('\u2705 Phase 2 Massey PCA functions defined (PC1 + PC2):')
print('   compute_massey_features()     -- Phase 1: mean percentile -> MASSEY')
print('   compute_massey_pca()          -- Phase 2: all-season PC1+PC2 -> MASSEY_PCA')
print('   compute_massey_pca_folds()    -- Phase 2: per-fold PC1+PC2  -> MASSEY_PCA_FOLDS')
print('   _get_massey_pca_diff()        -- Phase 2: (PC1_diff, PC2_diff) for matchup')


✅ Phase 2 Massey PCA functions defined (PC1 + PC2):
   compute_massey_features()     -- Phase 1: mean percentile -> MASSEY
   compute_massey_pca()          -- Phase 2: all-season PC1+PC2 -> MASSEY_PCA
   compute_massey_pca_folds()    -- Phase 2: per-fold PC1+PC2  -> MASSEY_PCA_FOLDS
   _get_massey_pca_diff()        -- Phase 2: (PC1_diff, PC2_diff) for matchup


**Define helper functions and the ensemble model trainer.**

The model is now a **LightGBM + logistic regression stacking ensemble** with **isotonic regression calibration**. The two models are trained on the same 24 features and their raw probability outputs are blended using `ENSEMBLE_WEIGHT` before calibration is applied.

The model predicts P(team1 wins) as a function of **24 features** (up from 14; expanded by A1 + A2):

| # | Feature | Description |
|---|---|---|
| 0 | `elo_diff` | Elo rating difference |
| 1 | `seed_diff` | Tournament seed difference |
| 2–7 | `fg_pct_diff` … `ast_diff` | Smoothed traditional box-score stat differences (6 features) |
| 8 | `off_eff_diff` | Offensive efficiency difference — points scored per 100 possessions (**A1**) |
| 9 | `def_eff_diff` | Defensive efficiency difference — points allowed per 100 possessions (**A1**) |
| 10 | `net_rating_diff` | Net rating difference = `off_eff − def_eff` (**A1**, replaces `net_eff`) |
| 11 | `efg_pct_diff` | Effective FG% difference — weights 3-pointers correctly (**A2**) |
| 12 | `tov_pct_diff` | Turnover rate difference — possession-normalised (**A2**) |
| 13 | `orb_pct_diff` | Offensive rebound rate difference (**A2**) |
| 14 | `ftr_diff` | Free-throw rate difference (FTA/FGA) (**A2**) |
| 15 | `def_efg_pct_diff` | Defensive effective FG% allowed difference (**A2**) |
| 16 | `def_tov_pct_diff` | Defensive turnover rate forced difference (**A2**) |
| 17 | `def_orb_pct_diff` | Opponent offensive rebound rate allowed difference (**A2**) |
| 18 | `def_ftr_diff` | Defensive free-throw rate allowed difference (**A2**) |
| 19 | `massey_pc1_diff` | Massey PCA consensus strength difference |
| 20 | `massey_pc2_diff` | Massey PCA style/tempo dimension difference |
| 21 | `elo_delta_diff` | Momentum: Elo gained during the season |
| 22 | `elo_diff × seed_diff` | Interaction: do Elo and seed agree? |
| 23 | `massey_pc1_diff × elo_diff` | Interaction: do both consensus signals reinforce? |

`_build_feature_row()` remains the single source of truth for feature construction — called by the trainer, walk-forward validation, the optimizer, and submission generation.

`_ensemble_predict_proba(X)` is the new single inference entry point: it blends LR + LGB outputs and applies the isotonic calibrator. `generate_submission` calls this instead of `MODEL.predict_proba` directly.

In [15]:
# Elo gained by tid during season-1 (momentum proxy)
def _get_elo_delta(elo_d, season, tid):
    end   = elo_d.get((season - 1, tid), ELO_INIT)
    start = ELO_REGRESS_ALPHA * elo_d.get((season - 2, tid), ELO_INIT) + (1 - ELO_REGRESS_ALPHA) * ELO_INIT
    return end - start

# Extract numeric seed from strings like 'W01', 'X16', etc.
def _parse_seed(seed_str):
    return int(seed_str[1:3])

# Return smoothed stat differences using previous season's stats.
# stat_keys order must match feature_names in train_prediction_model exactly.
def _get_stats_diff(season, t1_id, t2_id):
    prev = season - 1
    s1 = STATS.get((prev, t1_id), {})
    s2 = STATS.get((prev, t2_id), {})
    stat_keys = [
        # traditional (6)
        'fg_pct', 'fg3_pct', 'ft_pct', 'reb', 'to', 'ast',
        # A1: pace-adjusted efficiency (3)
        'off_eff', 'def_eff', 'net_rating',
        # A2: Four Factors offensive (4)
        'efg_pct', 'tov_pct', 'orb_pct', 'ftr',
        # A2: Four Factors defensive (4)
        'def_efg_pct', 'def_tov_pct', 'def_orb_pct', 'def_ftr',
    ]
    diffs = []
    for k in stat_keys:
        v1 = s1.get(k, float('nan'))
        v2 = s2.get(k, float('nan'))
        import math
        diffs.append(0.0 if (math.isnan(v1) or math.isnan(v2)) else v1 - v2)
    return diffs

def _build_feature_row(elo_diff, seed_diff, stat_diffs, mpca, mpca2, delta):
    return (
        [elo_diff, seed_diff]
        + stat_diffs                         # 17 stats (was 7; A1+A2 extended this)
        + [mpca, mpca2, delta,
           elo_diff * seed_diff,             # does Elo agree with seed consensus?
           mpca * elo_diff]                  # do both consensus signals reinforce?
    )

# C1: single inference entry point
def _ensemble_predict_proba(X_arr):
    """
    Blend LR + LGB raw probabilities, then apply isotonic calibration (C2).
    Returns a 1-D numpy array of calibrated probabilities.
    """
    lr_p  = MODEL.predict_proba(X_arr)[:, 1]
    lgb_p = LGB_MODEL.predict_proba(X_arr)[:, 1]
    raw   = ENSEMBLE_WEIGHT * lgb_p + (1.0 - ENSEMBLE_WEIGHT) * lr_p
    if CALIBRATOR is not None:
        return CALIBRATOR.transform(raw)
    return raw

# Train both models; store fitted objects in globals
def train_prediction_model() -> dict:
    global MODEL, LGB_MODEL, CALIBRATOR

    seed_map = {(r['Season'], r['TeamID']): _parse_seed(r['Seed'])
                for _, r in seeds.iterrows()}

    X, y = [], []
    for t_df in [m_tourney, w_tourney]:
        for _, row in t_df.iterrows():
            season = row['Season']
            if season < 2003: continue
            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo  = ELO.get((season-1, w_id), ELO_INIT)
            l_elo  = ELO.get((season-1, l_id), ELO_INIT)
            w_seed = seed_map.get((season, w_id), 8)
            l_seed = seed_map.get((season, l_id), 8)
            # lower teamID is always team 1, label=1 if they won
            if w_id < l_id:
                mpca, mpca2 = _get_massey_pca_diff(season, w_id, l_id)
                delta = _get_elo_delta(ELO, season, w_id) - _get_elo_delta(ELO, season, l_id)
                X.append(_build_feature_row(w_elo-l_elo, l_seed-w_seed,
                          _get_stats_diff(season, w_id, l_id), mpca, mpca2, delta))
                y.append(1)
            else:
                mpca, mpca2 = _get_massey_pca_diff(season, l_id, w_id)
                delta = _get_elo_delta(ELO, season, l_id) - _get_elo_delta(ELO, season, w_id)
                X.append(_build_feature_row(l_elo-w_elo, w_seed-l_seed,
                          _get_stats_diff(season, l_id, w_id), mpca, mpca2, delta))
                y.append(0)

    X, y = np.array(X), np.array(y)

    # Logistic Regression (existing) — kept for backward compat + ensemble component
    MODEL = LogisticRegression(C=LOGREG_C, solver='lbfgs', max_iter=500)
    MODEL.fit(X, y)

    # C1: LightGBM — train final model on full data
    print('  [C1] Training LightGBM on full dataset...')
    LGB_MODEL = lgb.LGBMClassifier(**LGB_PARAMS)
    LGB_MODEL.fit(X, y)

    # C1+C2: Compute OOF predictions for isotonic calibration.
    # OOF predictions prevent calibration leakage: the calibrator sees only
    # predictions on held-out folds the base model did NOT train on.
    print('  [C1+C2] Computing OOF predictions for isotonic calibration (5-fold)...')
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_lr = cross_val_predict(
        LogisticRegression(C=LOGREG_C, solver='lbfgs', max_iter=500),
        X, y, cv=kf, method='predict_proba'
    )[:, 1]

    oof_lgb = cross_val_predict(
        lgb.LGBMClassifier(**LGB_PARAMS),
        X, y, cv=kf, method='predict_proba'
    )[:, 1]

    oof_blend = ENSEMBLE_WEIGHT * oof_lgb + (1.0 - ENSEMBLE_WEIGHT) * oof_lr

    # C2: Fit isotonic regression calibrator on blended OOF scores
    CALIBRATOR = IsotonicRegression(out_of_bounds='clip')
    CALIBRATOR.fit(oof_blend, y)
    oof_brier = float(np.mean((oof_blend - y) ** 2))
    cal_brier  = float(np.mean((CALIBRATOR.transform(oof_blend) - y) ** 2))
    print(f'  [C2] OOF Brier before calibration : {oof_brier:.4f}')
    print(f'  [C2] OOF Brier after  calibration : {cal_brier:.4f}')

    # CV Brier on LR alone (kept for backward-compatible reporting)
    cv_probs = cross_val_score(
        LogisticRegression(C=LOGREG_C, solver='lbfgs', max_iter=500), X, y,
        scoring='neg_brier_score', cv=5
    )
    brier_lr = -cv_probs.mean()

    feature_names = [
        'elo_diff', 'seed_diff',
        # traditional stats (6)
        'fg_pct_diff', 'fg3_pct_diff', 'ft_pct_diff',
        'reb_diff', 'to_diff', 'ast_diff',
        # A1: pace-adjusted efficiency (3)
        'off_eff_diff', 'def_eff_diff', 'net_rating_diff',
        # A2: Four Factors offensive (4)
        'efg_pct_diff', 'tov_pct_diff', 'orb_pct_diff', 'ftr_diff',
        # A2: Four Factors defensive (4)
        'def_efg_pct_diff', 'def_tov_pct_diff', 'def_orb_pct_diff', 'def_ftr_diff',
        # ratings + momentum
        'massey_pc1_diff',
        'massey_pc2_diff',
        'elo_delta_diff',
        # interactions
        'elo_diff_x_seed_diff',
        'mpca_x_elo_diff',
    ]
    coefs = {name: f'{c:.6f}' for name, c in zip(feature_names, MODEL.coef_[0])}
    return {
        'status'               : 'success',
        'training_games'       : len(y),
        'features'             : feature_names,
        'num_features'         : len(feature_names),
        'cv_brier_lr'          : f'{brier_lr:.4f}',
        'oof_brier_ensemble'   : f'{oof_brier:.4f}',
        'oof_brier_calibrated' : f'{cal_brier:.4f}',
        'ensemble_weight_lgb'  : ENSEMBLE_WEIGHT,
        'logreg_C'             : LOGREG_C,
        'coefficients'         : coefs,
        'intercept'            : f'{MODEL.intercept_[0]:.6f}',
        'message'              : (
            f'Ensemble (LR + LGB, LGB weight={ENSEMBLE_WEIGHT}) trained on {len(y)} games, '
            f'{len(feature_names)} features. '
            f'OOF ensemble Brier: {oof_brier:.4f} -> calibrated: {cal_brier:.4f}'
        )
    }

print('✅ train_prediction_model defined (C1: LR + LightGBM ensemble, C2: isotonic calibration)')
print('✅ _ensemble_predict_proba defined (single inference entry point)')


✅ train_prediction_model defined (C1: LR + LightGBM ensemble, C2: isotonic calibration)
✅ _ensemble_predict_proba defined (single inference entry point)


**Define and run walk-forward validation.**

For each eval season from 2010–2025, both a logistic regression and a LightGBM classifier are trained on all tournament games *before* that season and evaluated on that season's games using the blended ensemble, which is exactly mirroring real deployment where future data is unavailable.

Seasons 2023–2025 serve as a holdout set: they fall within the validation loop but are reported separately to check for hyperparameter overfit. The Massey PCA features used in each fold are fit on training seasons only (`MASSEY_PCA_FOLDS`), preventing any data leakage.

Each fold trains both `LogisticRegression` and `lgb.LGBMClassifier` independently and blends their predictions using `ENSEMBLE_WEIGHT`. Isotonic calibration is **not** applied within folds — it is fit on full-data OOF predictions inside `train_prediction_model` to avoid instability on small per-fold tournament game counts (~60–130 games per season).

In [16]:
def walk_forward_validation(start_eval_season=2010, end_eval_season=2025):
    """
    For each season in [start_eval_season, end_eval_season]:
      - Train on all tournament games BEFORE that season
      - Evaluate on that season's games using LR + LGB ensemble
      - Record Brier score
    Mirrors real deployment: you never train on future data.
    """
    # ensure per-fold PCA scores are pre-computed
    if not MASSEY_PCA_FOLDS:
        print('  [Auto-computing per-fold Massey PCA scores -- run once...]')
        compute_massey_pca_folds()

    seed_map = {(r['Season'], r['TeamID']): _parse_seed(r['Seed'])
                for _, r in seeds.iterrows()}
    all_tourney   = pd.concat([m_tourney, w_tourney])
    season_scores = []

    for eval_season in range(start_eval_season, end_eval_season + 1):
        train_games = all_tourney[all_tourney['Season'] < eval_season]
        test_games  = all_tourney[all_tourney['Season'] == eval_season]
        if len(train_games) < 50 or len(test_games) == 0:
            continue

        # use per-fold PCA fitted only on seasons before eval_season
        pca_fold = MASSEY_PCA_FOLDS.get(eval_season, {})

        X_tr, y_tr = [], []
        for _, row in train_games.iterrows():
            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo  = ELO.get((row['Season'] - 1, w_id), ELO_INIT)
            l_elo  = ELO.get((row['Season'] - 1, l_id), ELO_INIT)
            w_seed = seed_map.get((row['Season'], w_id), 8)
            l_seed = seed_map.get((row['Season'], l_id), 8)
            if w_id < l_id:
                mpca, mpca2 = _get_massey_pca_diff(row['Season'], w_id, l_id, pca_fold)
                delta = _get_elo_delta(ELO, row['Season'], w_id) - _get_elo_delta(ELO, row['Season'], l_id)
                X_tr.append(_build_feature_row(w_elo - l_elo, l_seed - w_seed,
                            _get_stats_diff(row['Season'], w_id, l_id), mpca, mpca2, delta))
                y_tr.append(1)
            else:
                mpca, mpca2 = _get_massey_pca_diff(row['Season'], l_id, w_id, pca_fold)
                delta = _get_elo_delta(ELO, row['Season'], l_id) - _get_elo_delta(ELO, row['Season'], w_id)
                X_tr.append(_build_feature_row(l_elo - w_elo, w_seed - l_seed,
                            _get_stats_diff(row['Season'], l_id, w_id), mpca, mpca2, delta))
                y_tr.append(0)

        X_te, y_te = [], []
        for _, row in test_games.iterrows():
            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo  = ELO.get((row['Season'] - 1, w_id), ELO_INIT)
            l_elo  = ELO.get((row['Season'] - 1, l_id), ELO_INIT)
            w_seed = seed_map.get((row['Season'], w_id), 8)
            l_seed = seed_map.get((row['Season'], l_id), 8)
            if w_id < l_id:
                mpca, mpca2 = _get_massey_pca_diff(row['Season'], w_id, l_id, pca_fold)
                delta = _get_elo_delta(ELO, row['Season'], w_id) - _get_elo_delta(ELO, row['Season'], l_id)
                X_te.append(_build_feature_row(w_elo - l_elo, l_seed - w_seed,
                            _get_stats_diff(row['Season'], w_id, l_id), mpca, mpca2, delta))
                y_te.append(1)
            else:
                mpca, mpca2 = _get_massey_pca_diff(row['Season'], l_id, w_id, pca_fold)
                delta = _get_elo_delta(ELO, row['Season'], l_id) - _get_elo_delta(ELO, row['Season'], w_id)
                X_te.append(_build_feature_row(l_elo - w_elo, w_seed - l_seed,
                            _get_stats_diff(row['Season'], l_id, w_id), mpca, mpca2, delta))
                y_te.append(0)

        X_tr_arr, y_tr_arr = np.array(X_tr), np.array(y_tr)
        X_te_arr, y_te_arr = np.array(X_te), np.array(y_te)

        model_cv_lr = LogisticRegression(C=LOGREG_C, solver='lbfgs', max_iter=500)
        model_cv_lr.fit(X_tr_arr, y_tr_arr)
        lr_probs = model_cv_lr.predict_proba(X_te_arr)[:, 1]

        model_cv_lgb = lgb.LGBMClassifier(**LGB_PARAMS)
        model_cv_lgb.fit(X_tr_arr, y_tr_arr)
        lgb_probs = model_cv_lgb.predict_proba(X_te_arr)[:, 1]

        probs = ENSEMBLE_WEIGHT * lgb_probs + (1.0 - ENSEMBLE_WEIGHT) * lr_probs

        brier = np.mean((probs - y_te_arr) ** 2)
        season_scores.append({'season': eval_season, 'brier': brier, 'games': len(y_te)})
        print(f"  {eval_season}: Brier = {brier:.4f}  ({len(y_te)} games)")

    scores_df = pd.DataFrame(season_scores)
    print(f"\nMean walk-forward Brier (ensemble): {scores_df['brier'].mean():.4f}")
    print(f"Std across seasons:                  {scores_df['brier'].std():.4f}")
    print(f"Best season:  {scores_df.loc[scores_df['brier'].idxmin(), 'season']} ({scores_df['brier'].min():.4f})")
    print(f"Worst season: {scores_df.loc[scores_df['brier'].idxmax(), 'season']} ({scores_df['brier'].max():.4f})")

    holdout = scores_df[scores_df['season'].isin([2023, 2024, 2025])]
    if not holdout.empty:
        holdout_seasons = ', '.join(str(s) for s in holdout['season'].tolist())
        print(f"\nHoldout Brier ({holdout_seasons}): {holdout['brier'].mean():.4f}")
    return scores_df


wf_results = walk_forward_validation()

  [Auto-computing per-fold Massey PCA scores -- run once...]
  2010: Brier = 0.1861  (127 games)
  2011: Brier = 0.1849  (130 games)
  2012: Brier = 0.1720  (130 games)
  2013: Brier = 0.1869  (130 games)
  2014: Brier = 0.1897  (130 games)
  2015: Brier = 0.1596  (130 games)
  2016: Brier = 0.1924  (130 games)
  2017: Brier = 0.1673  (130 games)
  2018: Brier = 0.1856  (130 games)
  2019: Brier = 0.1604  (130 games)
  2021: Brier = 0.1928  (129 games)
  2022: Brier = 0.1982  (134 games)
  2023: Brier = 0.1905  (134 games)
  2024: Brier = 0.1639  (134 games)
  2025: Brier = 0.1445  (134 games)

Mean walk-forward Brier (ensemble): 0.1783
Std across seasons:                  0.0158
Best season:  2025 (0.1445)
Worst season: 2022 (0.1982)

Holdout Brier (2023, 2024, 2025): 0.1663


## 5. Hyperparameter Optimization

We use a **two-stage optimizer** to find the best parameters for the full pipeline.

**Stage 1 — Differential Evolution** searches over the 5 Elo and stats parameters (`ELO_K`, `ELO_HCA`, `ELO_RECENCY_ALPHA`, `STATS_SMOOTH_ALPHA`, `LOGREG_C`). Logistic regression is used as a fast proxy model. Runtime: ~3–4 hours.

**Stage 2 — Optuna TPE** searches over all 10 model parameters: `LOGREG_C`, `ENSEMBLE_WEIGHT`, and 8 LightGBM hyperparameters (`n_estimators`, `learning_rate`, `max_depth`, `num_leaves`, `reg_alpha`, `reg_lambda`, `subsample`, `colsample_bytree`). Because Elo and stats are precomputed once from Stage 1, each Optuna trial is ~5–10 seconds instead of ~45 seconds. Runtime: configurable timeout (default 3.5 hours).

### Why each parameter is (or isn't) optimized

| Parameter | Stage | Reason |
|---|---|---|
| `ELO_K` | 1 | Controls how fast ratings respond to results |
| `ELO_HCA` | 1 | Home-court advantage in Elo updates |
| `ELO_RECENCY_ALPHA` | 1 | Exponential recency weighting on K-factor |
| `STATS_SMOOTH_ALPHA` | 1 | Exponential smoothing weight on per-game stats |
| `ELO_INIT` | neither | Arbitrary baseline; only differences matter |
| `ELO_REGRESS_ALPHA` | neither | Hardcoded at 0.75; see Section 4 for rationale |
| `LOGREG_C` | 2 | L2 regularization strength for logistic regression |
| `ENSEMBLE_WEIGHT` | 2 | LGB blend weight; (1-weight) goes to LogReg |
| LGB `n_estimators` | 2 | Number of boosting rounds |
| LGB `learning_rate` | 2 | Step size shrinkage |
| LGB `max_depth` | 2 | Max tree depth |
| LGB `num_leaves` | 2 | Max leaves per tree |
| LGB `reg_alpha` | 2 | L1 regularization on leaf weights |
| LGB `reg_lambda` | 2 | L2 regularization on leaf weights |
| LGB `subsample` | 2 | Row subsampling fraction per tree |
| LGB `colsample_bytree` | 2 | Feature subsampling fraction per tree |

In [17]:
import time, optuna
from scipy.optimize import differential_evolution

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial noise

# STAGE 1: Differential Evolution bounds (Elo + stats params only)
STAGE1_BOUNDS = [
    (2.0,   30.0),   # ELO_K
    (0.0,  200.0),   # ELO_HCA
    (0.0,    2.0),   # ELO_RECENCY_ALPHA
    (0.05,  0.95),   # STATS_SMOOTH_ALPHA
    (0.01,  10.0),   # LOGREG_C  (kept here so Stage 1 LR proxy is well-tuned)
]
STAGE1_NAMES = ['ELO_K', 'ELO_HCA', 'ELO_RECENCY_ALPHA', 'STATS_SMOOTH_ALPHA', 'LOGREG_C']

# Optimizer-internal Elo recompute (side-effect-free)
def _opt_compute_elo(k, hca, recency_alpha):
    def _run(regular_df, tourney_df):
        elo = {}
        season_elos = {}
        all_games = pd.concat([regular_df, tourney_df]).sort_values(['Season', 'DayNum'])
        day_ranges = all_games.groupby('Season')['DayNum'].agg(min_day='min', max_day='max')
        prev_season = None
        for _, row in all_games.iterrows():
            season = row['Season']
            if season != prev_season and prev_season is not None:
                for tid, r in elo.items():
                    season_elos[(prev_season, tid)] = r
                elo = {tid: ELO_REGRESS_ALPHA * r + (1 - ELO_REGRESS_ALPHA) * ELO_INIT
                       for tid, r in elo.items()}
            prev_season = season
            min_d = day_ranges.loc[season, 'min_day']
            max_d = day_ranges.loc[season, 'max_day']
            t = (row['DayNum'] - min_d) / max(max_d - min_d, 1)
            K_game = k * np.exp(recency_alpha * t)
            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo = elo.get(w_id, ELO_INIT)
            l_elo = elo.get(l_id, ELO_INIT)
            w_loc = row.get('WLoc', 'N')
            w_adj = w_elo + (hca if w_loc == 'H' else (-hca if w_loc == 'A' else 0))
            exp_w = 1.0 / (1.0 + 10 ** ((l_elo - w_adj) / 400.0))
            margin          = row['WScore'] - row['LScore']
            winner_elo_diff = w_adj - l_elo
            K_game         *= mov_multiplier(margin, winner_elo_diff)
            elo[w_id] = w_elo + K_game * (1.0 - exp_w)
            elo[l_id] = l_elo + K_game * (0.0 - (1.0 - exp_w))
        if prev_season:
            for tid, r in elo.items():
                season_elos[(prev_season, tid)] = r
        return season_elos
    d = {}
    d.update(_run(m_regular, m_tourney))
    d.update(_run(w_regular, w_tourney))
    return d


# Optimizer-internal stats recompute — FIXED to include full A1+A2 stats
def _opt_compute_stats(stats_alpha):
    def _process(regular_df):
        shared = ['Season', 'DayNum']
        w_view = regular_df[shared + ['WTeamID', 'WScore', 'LScore',
                                       'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                                       'WOR', 'WDR', 'WTO', 'WAst',
                                       'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                                       'LOR', 'LDR', 'LTO']].copy()
        w_view.columns = shared + ['TeamID', 'Score', 'OppScore',
                                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                                    'OR', 'DR', 'TO', 'Ast',
                                    'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA',
                                    'OppOR', 'OppDR', 'OppTO']
        l_view = regular_df[shared + ['LTeamID', 'LScore', 'WScore',
                                       'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                                       'LOR', 'LDR', 'LTO', 'LAst',
                                       'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                                       'WOR', 'WDR', 'WTO']].copy()
        l_view.columns = shared + ['TeamID', 'Score', 'OppScore',
                                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                                    'OR', 'DR', 'TO', 'Ast',
                                    'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA',
                                    'OppOR', 'OppDR', 'OppTO']
        games = pd.concat([w_view, l_view], ignore_index=True)

        # Traditional stats
        games['fg_pct']  = games['FGM']  / games['FGA'].replace(0, np.nan)
        games['fg3_pct'] = games['FGM3'] / games['FGA3'].replace(0, np.nan)
        games['ft_pct']  = games['FTM']  / games['FTA'].replace(0, np.nan)
        games['reb']     = games['OR'] + games['DR']
        games['to']      = games['TO']
        games['ast']     = games['Ast']

        # A1: pace-adjusted efficiency
        games['poss']     = (games['FGA'] - games['OR'] + games['TO'] + 0.44 * games['FTA']).clip(lower=1)
        games['opp_poss'] = (games['OppFGA'] - games['OppOR'] + games['OppTO'] + 0.44 * games['OppFTA']).clip(lower=1)
        games['off_eff']  = games['Score']    / games['poss']     * 100
        games['def_eff']  = games['OppScore'] / games['opp_poss'] * 100
        games['net_rating'] = games['off_eff'] - games['def_eff']

        # A2: Four Factors offensive
        denom_tov = (games['FGA'] + 0.44 * games['FTA'] + games['TO']).replace(0, np.nan)
        denom_orb = (games['OR'] + games['OppDR']).replace(0, np.nan)
        games['efg_pct'] = (games['FGM'] + 0.5 * games['FGM3']) / games['FGA'].replace(0, np.nan)
        games['tov_pct'] = games['TO'] / denom_tov
        games['orb_pct'] = games['OR'] / denom_orb
        games['ftr']     = games['FTA'] / games['FGA'].replace(0, np.nan)

        # A2: Four Factors defensive
        opp_denom_tov = (games['OppFGA'] + 0.44 * games['OppFTA'] + games['OppTO']).replace(0, np.nan)
        opp_denom_orb = (games['OppOR'] + games['DR']).replace(0, np.nan)
        games['def_efg_pct'] = (games['OppFGM'] + 0.5 * games['OppFGM3']) / games['OppFGA'].replace(0, np.nan)
        games['def_tov_pct'] = games['OppTO'] / opp_denom_tov
        games['def_orb_pct'] = games['OppOR'] / opp_denom_orb
        games['def_ftr']     = games['OppFTA'] / games['OppFGA'].replace(0, np.nan)

        stat_cols = [
            'fg_pct', 'fg3_pct', 'ft_pct', 'reb', 'to', 'ast',
            'off_eff', 'def_eff', 'net_rating',
            'efg_pct', 'tov_pct', 'orb_pct', 'ftr',
            'def_efg_pct', 'def_tov_pct', 'def_orb_pct', 'def_ftr',
        ]
        games = games.sort_values(['Season', 'TeamID', 'DayNum'])
        smoothed = {}
        for (season, team_id), grp in games.groupby(['Season', 'TeamID']):
            row_d = {}
            for col in stat_cols:
                series = grp[col].dropna().tolist()
                row_d[col] = _safe_ses(series, alpha=stats_alpha) if series else np.nan
            smoothed[(season, team_id)] = row_d
        return smoothed

    d = {}
    d.update(_process(m_regular))
    d.update(_process(w_regular))
    return d


# Walk-forward for Stage 1 (LR proxy; full 17-stat feature set)
def _opt_walk_forward_lr(elo_d, stats_d, logreg_c, start=2006, end=2022):
    """
    Mean walk-forward Brier using LR only (fast proxy for Stage 1 Elo/stats tuning).
    Uses the full 17-stat feature set — identical to the actual model.
    """
    seed_map = {(r['Season'], r['TeamID']): _parse_seed(r['Seed'])
                for _, r in seeds.iterrows()}
    all_tourney = pd.concat([m_tourney, w_tourney])
    stat_keys = [
        'fg_pct', 'fg3_pct', 'ft_pct', 'reb', 'to', 'ast',
        'off_eff', 'def_eff', 'net_rating',
        'efg_pct', 'tov_pct', 'orb_pct', 'ftr',
        'def_efg_pct', 'def_tov_pct', 'def_orb_pct', 'def_ftr',
    ]

    def get_stats_diff(season, t1, t2):
        prev = season - 1
        s1 = stats_d.get((prev, t1), {})
        s2 = stats_d.get((prev, t2), {})
        return [0.0 if (np.isnan(s1.get(k, np.nan)) or np.isnan(s2.get(k, np.nan)))
                else s1.get(k, 0.0) - s2.get(k, 0.0) for k in stat_keys]

    briers = []
    for eval_season in range(start, end + 1):
        train_g = all_tourney[all_tourney['Season'] < eval_season]
        test_g  = all_tourney[all_tourney['Season'] == eval_season]
        if len(train_g) < 150 or len(test_g) == 0:
            continue
        pca_fold = MASSEY_PCA_FOLDS.get(eval_season, {})
        X_tr, y_tr, X_te, y_te = [], [], [], []
        for df, Xl, yl in [(train_g, X_tr, y_tr), (test_g, X_te, y_te)]:
            for _, row in df.iterrows():
                w_id, l_id = row['WTeamID'], row['LTeamID']
                w_elo  = elo_d.get((row['Season'] - 1, w_id), ELO_INIT)
                l_elo  = elo_d.get((row['Season'] - 1, l_id), ELO_INIT)
                w_seed = seed_map.get((row['Season'], w_id), 8)
                l_seed = seed_map.get((row['Season'], l_id), 8)
                if w_id < l_id:
                    mpca, mpca2 = _get_massey_pca_diff(row['Season'], w_id, l_id, pca_fold)
                    delta = _get_elo_delta(elo_d, row['Season'], w_id) - _get_elo_delta(elo_d, row['Season'], l_id)
                    Xl.append(_build_feature_row(w_elo - l_elo, l_seed - w_seed,
                              get_stats_diff(row['Season'], w_id, l_id), mpca, mpca2, delta))
                    yl.append(1)
                else:
                    mpca, mpca2 = _get_massey_pca_diff(row['Season'], l_id, w_id, pca_fold)
                    delta = _get_elo_delta(elo_d, row['Season'], l_id) - _get_elo_delta(elo_d, row['Season'], w_id)
                    Xl.append(_build_feature_row(l_elo - w_elo, w_seed - l_seed,
                              get_stats_diff(row['Season'], l_id, w_id), mpca, mpca2, delta))
                    yl.append(0)
        mdl = LogisticRegression(C=logreg_c, solver='lbfgs', max_iter=500)
        mdl.fit(np.array(X_tr), np.array(y_tr))
        probs = mdl.predict_proba(np.array(X_te))[:, 1]
        briers.append(np.mean((probs - np.array(y_te)) ** 2))
    return float(np.mean(briers)) if briers else 1.0


# Walk-forward for Stage 2 (ensemble; Elo/stats precomputed)
def _opt_walk_forward_ensemble(elo_d, stats_d, logreg_c, ensemble_weight, lgb_params,
                                start=2006, end=2022):
    """
    Mean walk-forward Brier using the full LR + LGB ensemble.
    Elo and stats are passed in (precomputed once in Stage 2), so this is fast.
    """
    seed_map = {(r['Season'], r['TeamID']): _parse_seed(r['Seed'])
                for _, r in seeds.iterrows()}
    all_tourney = pd.concat([m_tourney, w_tourney])
    stat_keys = [
        'fg_pct', 'fg3_pct', 'ft_pct', 'reb', 'to', 'ast',
        'off_eff', 'def_eff', 'net_rating',
        'efg_pct', 'tov_pct', 'orb_pct', 'ftr',
        'def_efg_pct', 'def_tov_pct', 'def_orb_pct', 'def_ftr',
    ]

    def get_stats_diff(season, t1, t2):
        prev = season - 1
        s1 = stats_d.get((prev, t1), {})
        s2 = stats_d.get((prev, t2), {})
        return [0.0 if (np.isnan(s1.get(k, np.nan)) or np.isnan(s2.get(k, np.nan)))
                else s1.get(k, 0.0) - s2.get(k, 0.0) for k in stat_keys]

    briers = []
    for eval_season in range(start, end + 1):
        train_g = all_tourney[all_tourney['Season'] < eval_season]
        test_g  = all_tourney[all_tourney['Season'] == eval_season]
        if len(train_g) < 150 or len(test_g) == 0:
            continue
        pca_fold = MASSEY_PCA_FOLDS.get(eval_season, {})
        X_tr, y_tr, X_te, y_te = [], [], [], []
        for df, Xl, yl in [(train_g, X_tr, y_tr), (test_g, X_te, y_te)]:
            for _, row in df.iterrows():
                w_id, l_id = row['WTeamID'], row['LTeamID']
                w_elo  = elo_d.get((row['Season'] - 1, w_id), ELO_INIT)
                l_elo  = elo_d.get((row['Season'] - 1, l_id), ELO_INIT)
                w_seed = seed_map.get((row['Season'], w_id), 8)
                l_seed = seed_map.get((row['Season'], l_id), 8)
                if w_id < l_id:
                    mpca, mpca2 = _get_massey_pca_diff(row['Season'], w_id, l_id, pca_fold)
                    delta = _get_elo_delta(elo_d, row['Season'], w_id) - _get_elo_delta(elo_d, row['Season'], l_id)
                    Xl.append(_build_feature_row(w_elo - l_elo, l_seed - w_seed,
                              get_stats_diff(row['Season'], w_id, l_id), mpca, mpca2, delta))
                    yl.append(1)
                else:
                    mpca, mpca2 = _get_massey_pca_diff(row['Season'], l_id, w_id, pca_fold)
                    delta = _get_elo_delta(elo_d, row['Season'], l_id) - _get_elo_delta(elo_d, row['Season'], w_id)
                    Xl.append(_build_feature_row(l_elo - w_elo, w_seed - l_seed,
                              get_stats_diff(row['Season'], l_id, w_id), mpca, mpca2, delta))
                    yl.append(0)
        X_tr_arr, y_tr_arr = np.array(X_tr), np.array(y_tr)
        X_te_arr, y_te_arr = np.array(X_te), np.array(y_te)

        lr_mdl = LogisticRegression(C=logreg_c, solver='lbfgs', max_iter=500)
        lr_mdl.fit(X_tr_arr, y_tr_arr)
        lr_p = lr_mdl.predict_proba(X_te_arr)[:, 1]

        lgb_mdl = lgb.LGBMClassifier(**lgb_params)
        lgb_mdl.fit(X_tr_arr, y_tr_arr)
        lgb_p = lgb_mdl.predict_proba(X_te_arr)[:, 1]

        probs = ensemble_weight * lgb_p + (1.0 - ensemble_weight) * lr_p
        briers.append(np.mean((probs - y_te_arr) ** 2))
    return float(np.mean(briers)) if briers else 1.0


# Stage 1: DE objective and state
_s1_state = {'n_evals': 0, 'best': float('inf'), 't_start': None}

def _s1_objective(params):
    k, hca, rec_alpha, stats_alpha, logreg_c = params
    _s1_state['n_evals'] += 1
    if _s1_state['t_start'] is None:
        _s1_state['t_start'] = time.time()

    t0 = time.time()
    elo_d   = _opt_compute_elo(k, hca, rec_alpha)
    stats_d = _opt_compute_stats(stats_alpha)
    score   = _opt_walk_forward_lr(elo_d, stats_d, logreg_c)
    elapsed = time.time() - t0

    n = _s1_state['n_evals']
    tag = ''
    if score < _s1_state['best']:
        _s1_state['best'] = score
        tag = '✨ NEW BEST'
    print(f"  S1 [{n:3d}] Brier={score:.5f} {tag}  "
          f"K={k:.1f} HCA={hca:.1f} RecA={rec_alpha:.3f} "
          f"StatsA={stats_alpha:.3f} C={logreg_c:.3f}  ({elapsed:.0f}s)")
    return score


# Main optimization function
def run_hyperparameter_optimization(
    stage1_popsize=4,
    stage1_maxiter=12,
    stage2_timeout_hours=3.5,
):
    """
    Two-stage hyperparameter optimization.

    Stage 1 — Differential Evolution over 5 Elo/stats parameters.
               Uses LR as a fast proxy model (~3-4 hours).
    Stage 2 — Optuna TPE over 10 model parameters (LOGREG_C, ENSEMBLE_WEIGHT,
               8 LGB hyperparameters). Elo/stats are precomputed once from
               Stage 1, so each trial is ~5-10s. Runs until timeout.

    Updates all globals and saves best_params.json.

    Args:
        stage1_popsize:       DE population multiplier (popsize × n_params candidates)
        stage1_maxiter:       DE max generations
        stage2_timeout_hours: Optuna time budget in hours
    """
    global ELO_K, ELO_HCA, ELO_RECENCY_ALPHA, STATS_SMOOTH_ALPHA
    global LOGREG_C, ENSEMBLE_WEIGHT, LGB_PARAMS

    # STAGE 1
    _s1_state['n_evals'] = 0
    _s1_state['best']    = float('inf')
    _s1_state['t_start'] = None

    max_evals_s1 = (stage1_popsize * len(STAGE1_BOUNDS)) * (1 + stage1_maxiter)
    print('=' * 65)
    print('STAGE 1: Elo + Stats — Differential Evolution')
    print('=' * 65)
    print(f'Parameters : {STAGE1_NAMES}')
    print(f'Max evals  : popsize({stage1_popsize}) x n_params({len(STAGE1_BOUNDS)}) x (1 + maxiter({stage1_maxiter})) = {max_evals_s1}')
    print(f'Proxy model: Logistic Regression (fast; LGB tuned in Stage 2)')
    print(f'Feature set: full 17-stat set (A1 + A2 included)')
    print(f'Opt window : 2006–2022  |  Holdout: 2023–2025')
    print('=' * 65)

    s1_result = differential_evolution(
        _s1_objective,
        bounds        = STAGE1_BOUNDS,
        seed          = 42,
        popsize       = stage1_popsize,
        maxiter       = stage1_maxiter,
        tol           = 1e-4,
        mutation      = (0.5, 1.0),
        recombination = 0.7,
        polish        = True,
        disp          = False,
        workers       = 1,
    )

    k_opt, hca_opt, rec_opt, stats_opt, c_stage1 = s1_result.x
    s1_time = time.time() - _s1_state['t_start']
    print(f'\nStage 1 complete ({_s1_state["n_evals"]} evals, {s1_time/60:.1f} min)')
    print(f'Best LR walk-forward Brier: {s1_result.fun:.5f}')
    for name, val in zip(STAGE1_NAMES, s1_result.x):
        print(f'  {name:22s} = {val:.4f}')

    # Precompute Elo and stats once with Stage 1 results — Stage 2 reuses these
    print('\nPrecomputing Elo and stats with Stage 1 params (done once for Stage 2)...')
    t_pre = time.time()
    elo_opt   = _opt_compute_elo(k_opt, hca_opt, rec_opt)
    stats_opt_d = _opt_compute_stats(stats_opt)
    print(f'Precompute done in {time.time()-t_pre:.0f}s.')

    # Stage 1 holdout (LR proxy — for reference)
    holdout_s1 = _opt_walk_forward_lr(elo_opt, stats_opt_d, c_stage1, start=2023, end=2025)
    print(f'Stage 1 holdout Brier (LR proxy, 2023-2025): {holdout_s1:.5f}')

    # STAGE 2
    print('\n' + '=' * 65)
    print('STAGE 2: Model Params — Optuna TPE')
    print('=' * 65)
    print(f'Parameters : LOGREG_C, ENSEMBLE_WEIGHT, 8 LGB hyperparameters')
    print(f'Elo/stats  : fixed from Stage 1 (precomputed)')
    print(f'Timeout    : {stage2_timeout_hours} hours')
    print('=' * 65)

    s2_state = {'n_trials': 0, 'best': float('inf'), 't_start': time.time()}

    def _s2_objective(trial):
        # LOGREG_C and ensemble weight
        logreg_c       = trial.suggest_float('logreg_c',        0.01,  10.0,  log=True)
        ensemble_w     = trial.suggest_float('ensemble_weight',  0.0,   1.0)

        # LGB hyperparameters
        n_estimators     = trial.suggest_int  ('n_estimators',     100,  600)
        learning_rate    = trial.suggest_float('learning_rate',     0.01, 0.20, log=True)
        max_depth        = trial.suggest_int  ('max_depth',         3,    7)
        num_leaves       = trial.suggest_int  ('num_leaves',        8,    63)
        reg_alpha        = trial.suggest_float('reg_alpha',         0.0,  5.0)
        reg_lambda       = trial.suggest_float('reg_lambda',        0.0,  5.0)
        subsample        = trial.suggest_float('subsample',         0.5,  1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree',  0.5,  1.0)

        trial_lgb_params = dict(
            n_estimators     = n_estimators,
            learning_rate    = learning_rate,
            max_depth        = max_depth,
            num_leaves       = num_leaves,
            reg_alpha        = reg_alpha,
            reg_lambda       = reg_lambda,
            subsample        = subsample,
            colsample_bytree = colsample_bytree,
            random_state     = 42,
            n_jobs           = -1,
            verbose          = -1,
        )

        score = _opt_walk_forward_ensemble(
            elo_opt, stats_opt_d, logreg_c, ensemble_w, trial_lgb_params
        )

        s2_state['n_trials'] += 1
        tag = ''
        if score < s2_state['best']:
            s2_state['best'] = score
            tag = '✨ NEW BEST'
        elapsed_total = (time.time() - s2_state['t_start']) / 60
        print(f"  S2 [{s2_state['n_trials']:4d}] Brier={score:.5f} {tag}  "
              f"C={logreg_c:.3f} w={ensemble_w:.2f} "
              f"n_est={n_estimators} lr={learning_rate:.3f} "
              f"depth={max_depth} leaves={num_leaves}  "
              f"({elapsed_total:.1f} min elapsed)")
        return score

    study = optuna.create_study(
        direction = 'minimize',
        sampler   = optuna.samplers.TPESampler(seed=42),
    )

    # Seed with current LGB_PARAMS so Stage 2 starts from a known-good point
    initial_params = {
        'logreg_c'        : c_stage1,
        'ensemble_weight' : ENSEMBLE_WEIGHT,
        'n_estimators'    : LGB_PARAMS['n_estimators'],
        'learning_rate'   : LGB_PARAMS['learning_rate'],
        'max_depth'       : LGB_PARAMS['max_depth'],
        'num_leaves'      : LGB_PARAMS['num_leaves'],
        'reg_alpha'       : LGB_PARAMS['reg_alpha'],
        'reg_lambda'      : LGB_PARAMS['reg_lambda'],
        'subsample'       : LGB_PARAMS['subsample'],
        'colsample_bytree': LGB_PARAMS['colsample_bytree'],
    }
    study.enqueue_trial(initial_params)

    study.optimize(
        _s2_objective,
        timeout   = stage2_timeout_hours * 3600,
        show_progress_bar = False,
    )

    best_s2 = study.best_params
    s2_time = time.time() - s2_state['t_start']
    print(f'\nStage 2 complete ({s2_state["n_trials"]} trials, {s2_time/60:.1f} min)')
    print(f'Best ensemble walk-forward Brier (2006-2022): {study.best_value:.5f}')
    print('Best Stage 2 params:')
    for k, v in best_s2.items():
        print(f'  {k:22s} = {v}')

    # Evaluate final ensemble on holdout with best Stage 2 params
    best_lgb_params = dict(
        n_estimators     = best_s2['n_estimators'],
        learning_rate    = best_s2['learning_rate'],
        max_depth        = best_s2['max_depth'],
        num_leaves       = best_s2['num_leaves'],
        reg_alpha        = best_s2['reg_alpha'],
        reg_lambda       = best_s2['reg_lambda'],
        subsample        = best_s2['subsample'],
        colsample_bytree = best_s2['colsample_bytree'],
        random_state     = 42,
        n_jobs           = -1,
        verbose          = -1,
    )
    holdout_final = _opt_walk_forward_ensemble(
        elo_opt, stats_opt_d,
        best_s2['logreg_c'], best_s2['ensemble_weight'],
        best_lgb_params, start=2023, end=2025
    )
    gap = holdout_final - study.best_value
    print(f'\nHoldout Brier (2023-2025, ensemble): {holdout_final:.5f}')
    print(f'Gap (holdout - opt window)          : {gap:+.5f}  '
          f'({"⚠️  possible overfit" if gap > 0.008 else "✅ stable"})')

    # Update globals
    ELO_K              = k_opt
    ELO_HCA            = hca_opt
    ELO_RECENCY_ALPHA  = rec_opt
    STATS_SMOOTH_ALPHA = stats_opt
    LOGREG_C           = best_s2['logreg_c']
    ENSEMBLE_WEIGHT    = best_s2['ensemble_weight']
    LGB_PARAMS.update(best_lgb_params)

    # Save to JSON
    best = dict(
        # Stage 1: Elo/stats
        ELO_K              = float(k_opt),
        ELO_HCA            = float(hca_opt),
        ELO_RECENCY_ALPHA  = float(rec_opt),
        STATS_SMOOTH_ALPHA = float(stats_opt),
        # Stage 2: model
        LOGREG_C           = float(best_s2['logreg_c']),
        ENSEMBLE_WEIGHT    = float(best_s2['ensemble_weight']),
        LGB_N_ESTIMATORS   = int(best_s2['n_estimators']),
        LGB_LEARNING_RATE  = float(best_s2['learning_rate']),
        LGB_MAX_DEPTH      = int(best_s2['max_depth']),
        LGB_NUM_LEAVES     = int(best_s2['num_leaves']),
        LGB_REG_ALPHA      = float(best_s2['reg_alpha']),
        LGB_REG_LAMBDA     = float(best_s2['reg_lambda']),
        LGB_SUBSAMPLE      = float(best_s2['subsample']),
        LGB_COLSAMPLE      = float(best_s2['colsample_bytree']),
        # Diagnostics
        s1_opt_brier       = float(s1_result.fun),
        s1_holdout_brier   = float(holdout_s1),
        s2_opt_brier       = float(study.best_value),
        s2_holdout_brier   = float(holdout_final),
    )
    import json as _json
    with open(PARAMS_PATH, 'w') as _f:
        _json.dump(best, _f, indent=2)

    print(f'\n💾 All parameters saved to {PARAMS_PATH}')
    print(f'✅ Globals updated. Run the pipeline cell to regenerate your submission.')
    return study


print('✅ run_hyperparameter_optimization defined (two-stage: DE + Optuna)')


✅ run_hyperparameter_optimization defined (two-stage: DE + Optuna)


## 6. Run Pipeline

In [18]:
# ── Run pipeline with current (or optimised) hyperparameters ───
# After run_hyperparameter_optimization() updates the globals,
# you MUST clear the old dicts and recompute everything.
# This cell does that correctly.

ELO.clear()
STATS.clear()
MASSEY.clear()
MASSEY_PCA.clear()    
MASSEY_PCA_FOLDS.clear()   

print('=== Step 1a: Recency-weighted Elo ===')
print(compute_elo_ratings())

print('\n=== Step 1b: Massey Ordinals mean-percentile ===')  
print(compute_massey_features())     

print('\n=== Step 1c: Massey PCA -- all-season fit ===')
print(compute_massey_pca())

print('\n=== Step 1d: Massey PCA -- per-fold pre-compute ===')
print(compute_massey_pca_folds())

print('\n=== Step 2: SES team stats ===')
print(compute_smoothed_stats())

print('\n=== Step 3: Walk-forward validation ===')
wf_results = walk_forward_validation()

print('\n=== Step 4: Train final model ===')
print(train_prediction_model())

print('\n=== Pipeline complete ===')


=== Step 1a: Recency-weighted Elo ===
{'status': 'success', 'total_ratings': 14452, 'top_mens': ['Duke: 2072', 'Arizona: 2039', 'Michigan: 2024', 'Florida: 1973', 'Houston: 1954'], 'top_womens': ['Connecticut: 2230', 'UCLA: 2204', 'Texas: 2146', 'South Carolina: 2144', 'LSU: 2053'], 'message': 'Recency-weighted Elo computed through 2026 (men) and 2026 (women).'}

=== Step 1b: Massey Ordinals mean-percentile ===
{'status': 'success', 'total_team_seasons': 7992, 'seasons_covered': '2003–2026', 'n_seasons': 23, 'systems_in_latest_season': 19, 'mean_percentile_latest': '0.5001  (should be ≈0.50)', 'message': 'Massey mean-percentile scores computed for 7,992 team-seasons across 23 seasons using 19 systems (latest season).'}

=== Step 1c: Massey PCA -- all-season fit ===
{'status': 'success', 'total_team_seasons': 7992, 'seasons_covered': '2003–2026', 'teams_in_latest': 365, 'message': 'Massey PCA (PC1+PC2) scores computed for 7,992 team-seasons across 23 seasons (all-season fit for final mo

## A. Hyperparameter Optimization

Two cells are available here depending on what you need to run:

- **Cell 40** — `run_hyperparameter_optimization()`: full two-stage optimizer (Stage 1 DE + Stage 2 Optuna). Use this when starting fresh or after major pipeline changes. Runtime ~7 hours.
- **Cell 41** — `run_stage2_standalone`: runs Stage 2 (Optuna) only, using Stage 1 params hardcoded from the overnight log. Use this when Stage 1 has already converged and only model parameters need tuning. Runtime ~3.5 hours.

In [19]:
# ════════════════════════════════════════════════════════════════
# TO RE-RUN OPTIMISATION (e.g. after adding new features):
#
#   result = run_hyperparameter_optimization()
#
# The function will:
#   1. Search for the best hyperparameters (~7 hours)
#   2. Update the global variables in memory
#   3. Save best_params.json to the project root
#
# Then run the pipeline cell above (Step 1–4) to retrain
# with the new optimal parameters.
# ════════════════════════════════════════════════════════════════

# Uncomment to run:
# result = run_hyperparameter_optimization()


In [20]:
# # Stage 2 — Independent Run (Stage 1 params recovered from overnight log)
# #
# # Stage 1 DE search completed all 260 evaluations and converged cleanly.
# # The polish step was cut off by Kaggle's session timeout before results
# # could be saved. Parameters below are taken directly from the overnight
# # log — the best result was eval 352, Brier=0.17595.
# #
# # This cell precomputes Elo and stats once from Stage 1 params (~45s),
# # then runs Optuna Stage 2 over all 10 model parameters.
# # It saves best_params.json on completion exactly as the full optimizer would.
# #
# # Prerequisites: run all cells top to bottom first so that
# # _opt_compute_elo, _opt_compute_stats, _opt_walk_forward_ensemble,
# # LGB_PARAMS, ENSEMBLE_WEIGHT, and PARAMS_PATH are all defined.

# import time, optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# # ── Stage 1 converged values (from overnight log) ────────────────────────────
# S1_K              = 19.3
# S1_HCA            = 112.3
# S1_REC_ALPHA      = 0.028
# S1_STATS_ALPHA    = 0.186
# S1_LOGREG_C       = 9.008   # starting point for Stage 2's C search
# S1_OPT_BRIER      = 0.17595

# # ── How long to run Stage 2 ──────────────────────────────────────────────────
# # 3.5 hours fits comfortably within Kaggle's 12-hour session limit.
# # Increase if you have more time available.
# STAGE2_TIMEOUT_HOURS = 3.5

# print("=" * 65)
# print("Stage 2 — Independent Run (Stage 1 params from overnight log)")
# print("=" * 65)
# print(f"  ELO_K              = {S1_K}")
# print(f"  ELO_HCA            = {S1_HCA}")
# print(f"  ELO_RECENCY_ALPHA  = {S1_REC_ALPHA}")
# print(f"  STATS_SMOOTH_ALPHA = {S1_STATS_ALPHA}")
# print(f"  LOGREG_C (S1)      = {S1_LOGREG_C}")
# print(f"  S1 opt Brier       = {S1_OPT_BRIER}")
# print(f"  Stage 2 timeout    = {STAGE2_TIMEOUT_HOURS} hours")
# print()

# # Precompute Elo and stats once — the only slow step here (~45s).
# # Every Optuna trial reuses these precomputed dicts, so each trial
# # only refits models (~5-10s each) instead of recomputing everything.
# print("Precomputing Elo and stats with Stage 1 params (one-time, ~45s)...")
# t0 = time.time()
# elo_s1   = _opt_compute_elo(S1_K, S1_HCA, S1_REC_ALPHA)
# stats_s1 = _opt_compute_stats(S1_STATS_ALPHA)
# print(f"Done in {time.time()-t0:.0f}s.\n")
# print("Starting Optuna Stage 2...\n")

# # ── Optuna objective ──────────────────────────────────────────────────────────
# s2_state = {'n_trials': 0, 'best': float('inf'), 't_start': time.time()}

# def _s2_objective_standalone(trial):
#     logreg_c       = trial.suggest_float('logreg_c',         0.01, 10.0, log=True)
#     ensemble_w     = trial.suggest_float('ensemble_weight',   0.0,  1.0)
#     n_estimators   = trial.suggest_int  ('n_estimators',      100,  600)
#     learning_rate  = trial.suggest_float('learning_rate',     0.01, 0.20, log=True)
#     max_depth      = trial.suggest_int  ('max_depth',         3,    7)
#     num_leaves     = trial.suggest_int  ('num_leaves',        8,    63)
#     reg_alpha      = trial.suggest_float('reg_alpha',         0.0,  5.0)
#     reg_lambda     = trial.suggest_float('reg_lambda',        0.0,  5.0)
#     subsample      = trial.suggest_float('subsample',         0.5,  1.0)
#     colsample      = trial.suggest_float('colsample_bytree',  0.5,  1.0)

#     trial_lgb = dict(
#         n_estimators     = n_estimators,
#         learning_rate    = learning_rate,
#         max_depth        = max_depth,
#         num_leaves       = num_leaves,
#         reg_alpha        = reg_alpha,
#         reg_lambda       = reg_lambda,
#         subsample        = subsample,
#         colsample_bytree = colsample,
#         random_state     = 42,
#         n_jobs           = -1,
#         verbose          = -1,
#     )

#     score = _opt_walk_forward_ensemble(
#         elo_s1, stats_s1, logreg_c, ensemble_w, trial_lgb,
#         start=2006, end=2022
#     )

#     s2_state['n_trials'] += 1
#     tag = ''
#     if score < s2_state['best']:
#         s2_state['best'] = score
#         tag = '✨ NEW BEST'
#     elapsed_min = (time.time() - s2_state['t_start']) / 60
#     print(f"  S2 [{s2_state['n_trials']:4d}] Brier={score:.5f} {tag}  "
#           f"C={logreg_c:.3f} w={ensemble_w:.2f} "
#           f"n_est={n_estimators} lr={learning_rate:.3f} "
#           f"depth={max_depth} leaves={num_leaves}  "
#           f"({elapsed_min:.1f} min elapsed)")
#     return score

# # ── Create study and seed with current defaults as trial 1 ───────────────────
# # Seeding ensures Optuna begins from a known-reasonable point rather than cold.
# study = optuna.create_study(
#     direction = 'minimize',
#     sampler   = optuna.samplers.TPESampler(seed=42),
# )
# study.enqueue_trial({
#     'logreg_c'        : S1_LOGREG_C,
#     'ensemble_weight' : ENSEMBLE_WEIGHT,
#     'n_estimators'    : LGB_PARAMS['n_estimators'],
#     'learning_rate'   : LGB_PARAMS['learning_rate'],
#     'max_depth'       : LGB_PARAMS['max_depth'],
#     'num_leaves'      : LGB_PARAMS['num_leaves'],
#     'reg_alpha'       : LGB_PARAMS['reg_alpha'],
#     'reg_lambda'      : LGB_PARAMS['reg_lambda'],
#     'subsample'       : LGB_PARAMS['subsample'],
#     'colsample_bytree': LGB_PARAMS['colsample_bytree'],
# })

# study.optimize(
#     _s2_objective_standalone,
#     timeout           = STAGE2_TIMEOUT_HOURS * 3600,
#     show_progress_bar = False,
# )

# # ── Results ───────────────────────────────────────────────────────────────────
# best_s2  = study.best_params
# s2_min   = (time.time() - s2_state['t_start']) / 60
# print(f'\nStage 2 complete ({s2_state["n_trials"]} trials, {s2_min:.1f} min)')
# print(f'Best ensemble Brier (2006-2022): {study.best_value:.5f}')
# print('Best Stage 2 params:')
# for k, v in best_s2.items():
#     print(f'  {k:22s} = {v}')

# # Holdout evaluation on 2023-2025 — never seen during optimization
# print('\nEvaluating holdout 2023-2025 (unseen during optimization)...')
# best_lgb = dict(
#     n_estimators     = best_s2['n_estimators'],
#     learning_rate    = best_s2['learning_rate'],
#     max_depth        = best_s2['max_depth'],
#     num_leaves       = best_s2['num_leaves'],
#     reg_alpha        = best_s2['reg_alpha'],
#     reg_lambda       = best_s2['reg_lambda'],
#     subsample        = best_s2['subsample'],
#     colsample_bytree = best_s2['colsample_bytree'],
#     random_state     = 42,
#     n_jobs           = -1,
#     verbose          = -1,
# )
# holdout_brier = _opt_walk_forward_ensemble(
#     elo_s1, stats_s1,
#     best_s2['logreg_c'], best_s2['ensemble_weight'],
#     best_lgb, start=2023, end=2025
# )
# gap = holdout_brier - study.best_value
# print(f'  Holdout Brier (2023-2025) : {holdout_brier:.5f}')
# print(f'  Opt-window Brier          : {study.best_value:.5f}')
# print(f'  Gap                       : {gap:+.5f}  '
#       f'({"⚠️  possible overfit" if gap > 0.008 else "✅ stable"})')

# # ── Update globals ────────────────────────────────────────────────────────────
# ELO_K              = S1_K
# ELO_HCA            = S1_HCA
# ELO_RECENCY_ALPHA  = S1_REC_ALPHA
# STATS_SMOOTH_ALPHA = S1_STATS_ALPHA
# LOGREG_C           = best_s2['logreg_c']
# ENSEMBLE_WEIGHT    = best_s2['ensemble_weight']
# LGB_PARAMS.update(best_lgb)

# # ── Save everything to best_params.json ──────────────────────────────────────
# import json as _json
# best_all = dict(
#     ELO_K              = float(S1_K),
#     ELO_HCA            = float(S1_HCA),
#     ELO_RECENCY_ALPHA  = float(S1_REC_ALPHA),
#     STATS_SMOOTH_ALPHA = float(S1_STATS_ALPHA),
#     LOGREG_C           = float(best_s2['logreg_c']),
#     ENSEMBLE_WEIGHT    = float(best_s2['ensemble_weight']),
#     LGB_N_ESTIMATORS   = int(best_s2['n_estimators']),
#     LGB_LEARNING_RATE  = float(best_s2['learning_rate']),
#     LGB_MAX_DEPTH      = int(best_s2['max_depth']),
#     LGB_NUM_LEAVES     = int(best_s2['num_leaves']),
#     LGB_REG_ALPHA      = float(best_s2['reg_alpha']),
#     LGB_REG_LAMBDA     = float(best_s2['reg_lambda']),
#     LGB_SUBSAMPLE      = float(best_s2['subsample']),
#     LGB_COLSAMPLE      = float(best_s2['colsample_bytree']),
#     s1_opt_brier       = float(S1_OPT_BRIER),
#     s2_opt_brier       = float(study.best_value),
#     s2_holdout_brier   = float(holdout_brier),
# )
# with open(PARAMS_PATH, 'w') as _f:
#     _json.dump(best_all, _f, indent=2)

# print(f'\n💾 All parameters saved to {PARAMS_PATH}')
# print(f'✅ Globals updated. Run the pipeline cell (Section 6) to regenerate your submission.')


In [29]:
# ============================================================
# Matchup prediction utilities + 2026 tournament evaluator
# ============================================================

from pathlib import Path
import json
import time
from uuid import uuid4

DEBUG_LOG_PATH = Path('debug-617998.log')


def _dbg_log(hypothesis_id, location, message, data, run_id='pre-fix'):
    # region agent log
    payload = {
        'sessionId': '617998',
        'runId': run_id,
        'hypothesisId': hypothesis_id,
        'id': f"log_{int(time.time() * 1000)}_{uuid4().hex[:8]}",
        'location': location,
        'message': message,
        'data': data,
        'timestamp': int(time.time() * 1000),
    }
    with DEBUG_LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
    # endregion


def _sim_one_game(season, team_a, team_b, seed_map):
    """
    Deterministic single-game prediction.

    Returns:
        winner_id, p_winner
    """
    lo, hi = min(team_a, team_b), max(team_a, team_b)

    e_lo = ELO.get((season - 1, lo), ELO_INIT)
    e_hi = ELO.get((season - 1, hi), ELO_INIT)
    s_lo = seed_map.get((season, lo), 8)
    s_hi = seed_map.get((season, hi), 8)

    stat_diff = _get_stats_diff(season, lo, hi)
    mpca_diff, mpca2_diff = _get_massey_pca_diff(season, lo, hi)
    elo_delta_diff = _get_elo_delta(ELO, season, lo) - _get_elo_delta(ELO, season, hi)

    features = np.array([
        _build_feature_row(
            e_lo - e_hi,
            s_hi - s_lo,
            stat_diff,
            mpca_diff,
            mpca2_diff,
            elo_delta_diff,
        )
    ])

    p_lo_wins = float(np.clip(_ensemble_predict_proba(features)[0], 0.01, 0.99))
    p_a_wins = p_lo_wins if team_a == lo else 1.0 - p_lo_wins

    winner = team_a if p_a_wins >= 0.5 else team_b
    p_winner = max(p_a_wins, 1.0 - p_a_wins)
    return winner, round(p_winner, 6)


def predict_matchup(team_a_name, team_b_name, season=CURRENT_SEASON, gender='M'):
    """Predict a named single matchup and print a compact summary."""
    teams_df = m_teams if gender == 'M' else w_teams
    seeds_df = m_seeds if gender == 'M' else w_seeds

    def _find_team(name_query):
        q = str(name_query).strip()
        if not q:
            raise ValueError("Team name query is empty.")
        exact_mask = teams_df["TeamName"].str.lower() == q.lower()
        matches = teams_df[exact_mask]
        if len(matches) == 0:
            mask = teams_df["TeamName"].str.contains(q, case=False, regex=False)
            matches = teams_df[mask]
        if len(matches) == 0:
            raise ValueError(f"No team found matching '{name_query}'.")
        if len(matches) > 1:
            options = matches["TeamName"].tolist()
            raise ValueError(f"Multiple teams match '{name_query}': {options}. Be more specific.")
        return int(matches.iloc[0]["TeamID"]), matches.iloc[0]["TeamName"]

    id_a, name_a = _find_team(team_a_name)
    id_b, name_b = _find_team(team_b_name)

    seed_map = {
        (int(r['Season']), int(r['TeamID'])): _parse_seed(r['Seed'])
        for _, r in seeds_df.iterrows()
    }

    winner_id, p_winner = _sim_one_game(season, id_a, id_b, seed_map)
    winner_name = name_a if winner_id == id_a else name_b

    print(f"Matchup  : {name_a} vs {name_b}")
    print(f"Winner   : {winner_name}")
    print(f"Win prob : {p_winner:.1%}")

    return {
        'season': season,
        'team_a': name_a,
        'team_b': name_b,
        'team_a_id': id_a,
        'team_b_id': id_b,
        'winner_id': winner_id,
        'winner_name': winner_name,
        'p_winner': p_winner,
    }


def _normalize_team_key(name):
    return ''.join(ch.lower() for ch in str(name) if ch.isalnum())


def _build_team_lookup_for_2026(gender='M'):
    teams_df = m_teams if gender == 'M' else w_teams
    lookup = {}
    for _, row in teams_df.iterrows():
        key = _normalize_team_key(row['TeamName'])
        lookup[key] = int(row['TeamID'])

    # Manual aliases for bracket-style spellings.
    aliases = {
        'stmarys': "St Mary's CA",
        'saintmarys': "St Mary's CA",
        'saintlouis': 'St Louis',
        'miamiohio': 'Miami OH',
        'miamifl': 'Miami FL',
        'northdakotastate': 'N Dakota St',
        'longislanduniversity': 'LIU Brooklyn',
        'universityofnortherniowa': 'Northern Iowa',
        'uni': 'Northern Iowa',
        'hawaii': 'Hawaii',
        'mcneese': 'McNeese St',
        'kennesawstate': 'Kennesaw',
        'queens': 'Queens NC',
        'tennesseestate': 'Tennessee St',
        'wrightstate': 'Wright St',
        'utahstate': 'Utah St',
        'iowastate': 'Iowa St',
        'michiganstate': 'Michigan St',
        'ohiostate': 'Ohio St',
        'prairieviewam': 'Prairie View',
        'uconn': 'Connecticut',
    }

    name_to_id = dict(zip(teams_df['TeamName'], teams_df['TeamID']))
    applied_aliases = []
    missing_aliases = []
    for alias_key, canonical_name in aliases.items():
        if canonical_name in name_to_id:
            lookup[alias_key] = int(name_to_id[canonical_name])
            applied_aliases.append(alias_key)
        else:
            missing_aliases.append({'alias': alias_key, 'canonical': canonical_name})

    # region agent log
    _dbg_log(
        'H1_alias_map_not_applied',
        'march-machine-learning-mania.ipynb:_build_team_lookup_for_2026',
        'built_team_lookup',
        {
            'gender': gender,
            'lookup_size': len(lookup),
            'has_saintmarys_alias': 'saintmarys' in lookup,
            'has_stmarys_alias': 'stmarys' in lookup,
            'applied_aliases_count': len(applied_aliases),
            'missing_aliases_count': len(missing_aliases),
        },
    )
    # endregion

    return lookup


def _resolve_team_id(name, lookup):
    key = _normalize_team_key(name)
    if key in lookup:
        # region agent log
        if "saintmary" in key:
            _dbg_log(
                'H2_resolution_path_issue',
                'march-machine-learning-mania.ipynb:_resolve_team_id',
                'resolved_direct_lookup',
                {'input_name': name, 'normalized_key': key, 'team_id': int(lookup[key])},
            )
        # endregion
        return int(lookup[key])

    for candidate_key, team_id in lookup.items():
        if key in candidate_key or candidate_key in key:
            # region agent log
            if "saintmary" in key:
                _dbg_log(
                    'H2_resolution_path_issue',
                    'march-machine-learning-mania.ipynb:_resolve_team_id',
                    'resolved_fuzzy_lookup',
                    {
                        'input_name': name,
                        'normalized_key': key,
                        'matched_candidate_key': candidate_key,
                        'team_id': int(team_id),
                    },
                )
            # endregion
            return int(team_id)

    # region agent log
    _dbg_log(
        'H3_unresolved_name_edge_case',
        'march-machine-learning-mania.ipynb:_resolve_team_id',
        'failed_to_resolve_team',
        {'input_name': name, 'normalized_key': key, 'lookup_size': len(lookup)},
    )
    # endregion
    raise ValueError(f"Unable to resolve team name '{name}' to TeamID.")


def evaluate_2026_mens_actual_matchups(
    actual_csv='data/actual_2026_mens_games.csv',
    output_csv='data/predictions_vs_actual_2026.csv',
    season=2026,
    verbose=True,
):
    """
    Evaluate deterministic predictions for each actual men's 2026 tournament game.

    Required columns in actual_csv:
        game_id, round, team_a, team_b, winner
    """
    if MODEL is None or LGB_MODEL is None:
        raise RuntimeError('Models are not trained. Run the pipeline training cell first.')

    actual_path = Path(actual_csv)
    if not actual_path.exists():
        raise FileNotFoundError(f'Missing actual results file: {actual_path}')

    actual_df = pd.read_csv(actual_path)
    required_cols = {'game_id', 'round', 'team_a', 'team_b', 'winner'}
    missing = required_cols - set(actual_df.columns)
    if missing:
        raise ValueError(f'actual results CSV missing columns: {sorted(missing)}')

    team_lookup = _build_team_lookup_for_2026(gender='M')
    m_name_lookup = dict(zip(m_teams['TeamID'], m_teams['TeamName']))
    # region agent log
    _dbg_log(
        'H4_stale_function_or_cell_order',
        'march-machine-learning-mania.ipynb:evaluate_2026_mens_actual_matchups',
        'evaluation_start',
        {
            'season': season,
            'actual_csv': str(actual_path),
            'rows_in_actual_csv': int(len(actual_df)),
            'contains_saintmarys_alias_now': 'saintmarys' in team_lookup,
        },
    )
    # endregion
    seed_map = {
        (int(r['Season']), int(r['TeamID'])): _parse_seed(r['Seed'])
        for _, r in m_seeds.iterrows()
    }

    rows = []
    unresolved = []

    for _, row in actual_df.iterrows():
        game_id = int(row['game_id'])
        round_name = str(row['round'])
        team_a_name = str(row['team_a'])
        team_b_name = str(row['team_b'])
        winner_name = str(row['winner'])

        try:
            team_a_id = _resolve_team_id(team_a_name, team_lookup)
            team_b_id = _resolve_team_id(team_b_name, team_lookup)
            actual_winner_id = _resolve_team_id(winner_name, team_lookup)
        except ValueError as exc:
            unresolved.append((game_id, str(exc)))
            continue

        pred_winner_id, pred_p_winner = _sim_one_game(season, team_a_id, team_b_id, seed_map)

        if pred_winner_id == team_a_id:
            pred_loser_id = team_b_id
            pred_p_team_a = pred_p_winner
        else:
            pred_loser_id = team_a_id
            pred_p_team_a = 1.0 - pred_p_winner

        actual_label_team_a = 1 if actual_winner_id == team_a_id else 0
        brier = (pred_p_team_a - actual_label_team_a) ** 2
        pred_correct = int(pred_winner_id == actual_winner_id)

        rows.append({
            'game_id': game_id,
            'season': season,
            'round': round_name,
            'team_a_id': team_a_id,
            'team_a_name': m_name_lookup.get(team_a_id, team_a_name),
            'team_b_id': team_b_id,
            'team_b_name': m_name_lookup.get(team_b_id, team_b_name),
            'actual_winner_id': actual_winner_id,
            'actual_winner_name': m_name_lookup.get(actual_winner_id, winner_name),
            'pred_winner_id': pred_winner_id,
            'pred_winner_name': m_name_lookup.get(pred_winner_id, 'Unknown'),
            'pred_loser_id': pred_loser_id,
            'pred_loser_name': m_name_lookup.get(pred_loser_id, 'Unknown'),
            'pred_p_winner': float(pred_p_winner),
            'pred_p_team_a': float(pred_p_team_a),
            'actual_label_team_a': int(actual_label_team_a),
            'pred_correct': pred_correct,
            'brier': float(brier),
        })

    if unresolved:
        # region agent log
        _dbg_log(
            'H5_specific_game_payload_mismatch',
            'march-machine-learning-mania.ipynb:evaluate_2026_mens_actual_matchups',
            'unresolved_names_found',
            {'count': len(unresolved), 'items': unresolved[:10]},
        )
        # endregion
        unresolved_msg = '\n'.join(f"  game {gid}: {msg}" for gid, msg in unresolved)
        raise ValueError('Unresolved team names in actual results CSV:\n' + unresolved_msg)

    out_df = pd.DataFrame(rows).sort_values('game_id').reset_index(drop=True)

    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)

    overall_acc = out_df['pred_correct'].mean() if len(out_df) else float('nan')
    overall_brier = out_df['brier'].mean() if len(out_df) else float('nan')

    if verbose:
        print('=' * 62)
        print('2026 MEN\'S TOURNAMENT — PREDICTIONS VS ACTUAL RESULTS')
        print('=' * 62)
        print(f'Total games evaluated : {len(out_df)}')
        print(f'Overall accuracy      : {overall_acc:.2%}')
        print(f'Overall Brier score   : {overall_brier:.5f}')

        print('\nRound-level summary:')
        round_summary = (
            out_df.groupby('round', as_index=False)
            .agg(games=('game_id', 'count'), accuracy=('pred_correct', 'mean'), brier=('brier', 'mean'))
        )
        round_summary['accuracy'] = round_summary['accuracy'].map(lambda x: f'{x:.2%}')
        round_summary['brier'] = round_summary['brier'].map(lambda x: f'{x:.5f}')
        print(round_summary.to_string(index=False))

        print(f"\nSaved detailed comparison CSV: {out_path}")

    return out_df


print('Added: _sim_one_game(), predict_matchup(), evaluate_2026_mens_actual_matchups()')
print("Usage:")
print("  predict_matchup('Duke', 'Kentucky', season=2026)")
print("  eval_df = evaluate_2026_mens_actual_matchups()")

Added: _sim_one_game(), predict_matchup(), evaluate_2026_mens_actual_matchups()
Usage:
  predict_matchup('Duke', 'Kentucky', season=2026)
  eval_df = evaluate_2026_mens_actual_matchups()


In [ ]:
predict_matchup('Houston', 'Duke', season=2026)

Matchup  : Houston vs Duke
Winner   : Duke
Win prob : 50.5%


{'season': 2026,
 'team_a': 'Houston',
 'team_b': 'Duke',
 'team_a_id': 1222,
 'team_b_id': 1181,
 'winner_id': 1181,
 'winner_name': 'Duke',
 'p_winner': 0.504762}

: 

In [28]:
# Run full deterministic comparison for all 67 actual 2026 men's tournament games
# Prerequisite: run the training pipeline cell first so MODEL/LGB_MODEL are fitted.

eval_2026_df = evaluate_2026_mens_actual_matchups(
    actual_csv='data/actual_2026_mens_games.csv',
    output_csv='data/predictions_vs_actual_2026.csv',
    season=2026,
    verbose=True,
)

eval_2026_df.head()

2026 MEN'S TOURNAMENT — PREDICTIONS VS ACTUAL RESULTS
Total games evaluated : 67
Overall accuracy      : 74.63%
Overall Brier score   : 0.17114

Round-level summary:
       round  games accuracy   brier
Championship      1    0.00% 0.25478
     Elite 8      4   75.00% 0.17035
  Final Four      2   50.00% 0.20456
  First Four      4   50.00% 0.27101
 Round of 32     16   87.50% 0.18723
 Round of 64     32   78.12% 0.13004
    Sweet 16      8   62.50% 0.23502

Saved detailed comparison CSV: data\predictions_vs_actual_2026.csv


,game_id,season,round,team_a_id,team_a_name,team_b_id,team_b_name,actual_winner_id,actual_winner_name,pred_winner_id,pred_winner_name,pred_loser_id,pred_loser_name,pred_p_winner,pred_p_team_a,actual_label_team_a,pred_correct,brier
0,1,2026,First Four,1224,Howard,1420,UMBC,1224,Howard,1224,Howard,1420,UMBC,0.504762,0.504762,1,1,0.245261
1,2,2026,First Four,1400,Texas,1301,NC State,1400,Texas,1400,Texas,1301,NC State,0.578313,0.578313,1,1,0.177820
2,3,2026,First Four,1341,Prairie View,1250,Lehigh,1341,Prairie View,1250,Lehigh,1341,Prairie View,0.571429,0.428571,1,0,0.326531
3,4,2026,First Four,1275,Miami OH,1374,SMU,1275,Miami OH,1374,SMU,1275,Miami OH,0.578313,0.421687,1,0,0.334446
4,5,2026,Round of 64,1395,TCU,1326,Ohio St,1395,TCU,1326,Ohio St,1395,TCU,0.504762,0.495238,1,0,0.254785
